# Local–global EEG: primary analysis notebook

Every analysis in the Results and every figure, in one place, starting from the saved derivatives.

| Section | What it does | Figure files |
|---|---|---|
| 0 | configuration, shared functions (inline) | — |
| 1 | MMN / P3b window tests, clusters, 2 × 2 ANOVA, reference control | `Figure1`, `Figure1_lean`, `Figure1_DE` |
| 2 | learners, cross-validated model comparison, recovery, surprise–amplitude relations, reset | `Figure2_schematic`, `Figure2_results`, `Figure2_recovery` |
| 3 | eLORETA cluster, auditory-cortex time course, spread checks, global null | `Figure3` |
| 4 | DCM: tables from the SPM outputs, DCM figures | `Figure_DCM_windows`, `Figure_DCM_robust`, `Figure_DCM`, `DCM_predicted_ERPs` |
| 5 | validation of every method on simulated data | — |
| 6 | every Results number vs this notebook | — |
| 7 | figure inventory | — |

**Not in this notebook:** preprocessing (`scripts/01`–`02`), the single-trial exports (`07`), the full
model-comparison and recovery fits (`06`; loaded here unless `RECOMPUTE_ALL_CV = True`), source extraction
(`11`, `18 --extract`) and the DCM fits (MATLAB, `scripts/dcm/`).

**How the figure cells work.** Each figure's code is the current figure script, placed inline inside one function
(`make_figure1()`, …) so its settings cannot change the notebook's variables. Figures are written to `figures/`
(PDF/SVG with editable text, PNG) and shown below each cell. If an input is missing on this computer (e.g. the
template head for Figure 3, or the DCM fits), that figure reports the problem and the notebook continues.
To change a figure, edit its script and rebuild this notebook (`python notebooks/build_primary_notebook.py`).

Run top to bottom with the `lg-eeg` environment. Total time: about 15–20 minutes.

In [ ]:
# ---------------------------------------------------------------- configuration
from pathlib import Path
import json, re, warnings, itertools, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, integrate
from scipy.io import loadmat
from scipy.special import digamma, gammaln, logsumexp
import pingouin as pg
import mne

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(context="notebook", style="ticks")

# project root: this notebook lives in <root>/notebooks/
ROOT = Path.cwd().parent if (Path.cwd().parent / "derivatives").exists() else Path.cwd()
ERP_DIR = ROOT / "derivatives" / "header1_erps"        # per-participant condition averages (07)
MC_DIR = ROOT / "derivatives" / "model_comparison"     # single-trial amplitudes, CV scores, recovery (06)
RV_DIR = ROOT / "derivatives" / "reset_validation"     # block-reset tests (05)
SRC_DIR = ROOT / "derivatives" / "source"              # template head, noise covariances (11)
SUBJECTS_DIR = SRC_DIR / "subjects"
BIDS_ROOT = ROOT / "data" / "bids"

# switches for the expensive steps (False = load the saved outputs)
RECOMPUTE_ALL_CV = False        # refit all 15 x 6 models x 2 windows (minutes)
RECOMPUTE_SOURCE = False        # recompute the eLORETA t-map + cluster test (~1 min); False = local_tmap.npz
RUN_SPREAD_CHECK = True         # frontal-generator latency + leakage simulation (~2 min)

# a-priori regions of interest and windows (Rutiku et al., 2024)
ROI = {"local": dict(chs=["Fz", "FC1", "FC2", "F1", "F2", "Cz"], win=(0.08, 0.14), dev="LD", std="LS"),
       "global": dict(chs=["Pz", "P1", "P2", "CPz", "POz", "CP1", "CP2"], win=(0.30, 0.70), dev="GD", std="GS")}
CONDS = ["LSGS", "LDGS", "LSGD", "LDGD"]
MODELS = ["M0 categories", "M1 adaptation", "M2 local", "M3 global", "M4 local+global", "M5 adapt+global"]
N_PERM, SEED = 2000, 42
SOUNDS = {1: "A", 2: "B"}
print("project root:", ROOT)

## 0. Shared functions

All functions are defined here, once, and used by every section. Each mirrors the corresponding project script
(named in its docstring) so the two can be compared line by line.

In [ ]:
# ---------------------------------------------------------------- data loading
def load_erps():
    '''Per-participant condition averages from 07_header1_erps.py.
    Returns times (s), channel names, and a dict cond -> array (subjects x channels x time) in µV
    (average reference, 30 Hz low-pass, baseline -100..0 ms).'''
    files = sorted(ERP_DIR.glob("sub-*.npz"))
    z0 = np.load(files[0])
    D = {c: np.stack([np.load(f)[f"topo__{c}"] for f in files]) for c in ["LS", "LD", "GS", "GD"]}
    return z0["times"], list(z0["ch_names"]), D, [f.stem for f in files]


def roi_wave(D, cond, chs, roi_chs):
    '''ROI average: subjects x time.'''
    return D[cond][:, [chs.index(c) for c in roi_chs]].mean(1)


def window_test(diff, times, win):
    '''Window mean of a difference wave per participant -> one-sample t-test vs 0.
    Returns mean, 95% CI half-width, t, p, number of participants with the group sign.'''
    w = (times >= win[0]) & (times <= win[1])
    m = diff[:, w].mean(1)
    t = stats.ttest_1samp(m, 0)
    ci = stats.t.ppf(0.975, len(m) - 1) * m.std(ddof=1) / np.sqrt(len(m))
    return dict(mean=m.mean(), ci95=ci, t=t.statistic, p=t.pvalue, same_sign=int((np.sign(m) == np.sign(m.mean())).sum())), m


def cluster_1d(diff, times, tmin=0.0, tmax=0.7, n_perm=N_PERM, seed=SEED):
    '''Temporal cluster permutation test (sign-flip) on a difference wave, as in 14_figure1.py.
    Cluster-forming threshold = two-sided p < .05 for t(n-1).'''
    sel = (times >= tmin) & (times <= tmax)
    n = len(diff)
    T, clusters, pv, _ = mne.stats.permutation_cluster_1samp_test(
        diff[:, sel], threshold=stats.t.ppf(0.975, n - 1), n_permutations=n_perm, tail=0, seed=seed,
        out_type="indices", verbose=False)
    ts = times[sel]
    rows = [dict(start_ms=ts[c[0]].min() * 1e3, end_ms=ts[c[0]].max() * 1e3, p=p,
                 sign="+" if T[c[0]].mean() > 0 else "-") for c, p in zip(clusters, pv) if p < 0.05]
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- stimulus sequence + learners
def load_sequence(subject):
    '''Quintlet table (one row per presented quintlet) and sound onset times (PTB clock, s),
    from the sourcedata SEQUENCE / critical-events files (as 06_model_comparison.py).'''
    src = BIDS_ROOT / "sourcedata" / subject / "ses-LGeffect"
    seq = loadmat(next(src.glob("GlobLoc_stimulation_SEQUENCE*.mat")), squeeze_me=True)["SEQUENCE"]
    tz = loadmat(next(src.glob("GlobLoc_critical_events*.mat")), squeeze_me=True)["tz"]
    tz = tz[tz > 0]
    rows = []
    for b, blk in enumerate(seq, start=1):
        blk = np.atleast_2d(blk).astype(int)
        std = "".join(SOUNDS[x] for x in blk[0])          # first quintlet of a block = its standard
        for i, q in enumerate(blk):
            rows.append(dict(block=b, trial_in_block=i, quintlet="".join(SOUNDS[x] for x in q), block_standard=std))
    qs = pd.DataFrame(rows)
    qs.insert(0, "seq_idx", np.arange(len(qs)))
    assert len(tz) == 5 * len(qs), "one onset per sound expected"
    return qs, tz


def leaky_surprise(outcomes, half_life, resets):
    '''Leaky Beta-Bernoulli learner (06/07/17/25).
    outcomes: 0/1 per observation; prior Beta(1,1); counts decay by 2^(-1/half_life) per observation
    (half_life = inf: no forgetting); counts set to 0 where resets is True (block start).
    Returns the surprise -log2 p(observed outcome) computed BEFORE updating on it.'''
    lam = 1.0 if np.isinf(half_life) else 2.0 ** (-1.0 / half_life)
    n1 = n0 = 0.0
    S = np.empty(len(outcomes))
    for t, y in enumerate(outcomes):
        if resets is not None and resets[t]:
            n1 = n0 = 0.0
        mu = (n1 + 1) / (n1 + n0 + 2)                     # posterior mean p(change)
        S[t] = -np.log2(mu if y else 1 - mu)
        n1, n0 = lam * n1 + y, lam * n0 + (1 - y)
    return S


def adaptation_trace(times, ids, tau):
    '''Stimulus-specific adaptation (M1): each sound adds 1 to its own trace, all traces decay as
    exp(-dt/tau); the value read out is the trace of the sound just BEFORE it is played.'''
    a = {"A": 0.0, "B": 0.0}
    last, out = times[0], np.empty(len(times))
    for i, (t, k) in enumerate(zip(times, ids)):
        dec = np.exp(-(t - last) / tau)
        a["A"] *= dec; a["B"] *= dec
        out[i] = a[k]
        a[k] += 1.0
        last = t
    return out


def learner_outcomes(qs):
    '''Outcome streams for the two learners.
    Local (M2): every sound-to-sound transition (4 per quintlet), 1 = change; only 4->5 can change here.
    Global (M3): one outcome per quintlet, 1 = 5th sound differs from the first four.'''
    first, fifth = qs["quintlet"].str[0].to_numpy(), qs["quintlet"].str[4].to_numpy()
    start = (qs["trial_in_block"] == 0).to_numpy()
    loc = np.zeros((len(qs), 4), int); loc[:, 3] = fifth != first
    loc_reset = np.zeros(4 * len(qs), bool); loc_reset[::4] = start
    return loc.ravel(), loc_reset, (fifth != first).astype(int), start


def learner_surprise(subject, clocks):
    '''Per-quintlet local (S_loc) and global (S_glob) surprise with the participant's own selected
    parameters (M2 chosen on the early window, M3 on the late window; clocks.tsv from 06).'''
    qs, _ = load_sequence(subject)
    c = clocks[clocks["subject"] == subject]
    p_loc = c[(c["component"] == "local_80-140ms") & (c["model"] == "M2 local")].iloc[0]
    p_glo = c[(c["component"] == "global_300-700ms") & (c["model"] == "M3 global")].iloc[0]
    lo, lr, go, gr = learner_outcomes(qs)
    qs["S_loc"] = leaky_surprise(lo, float(p_loc["param"]), lr if bool(p_loc["reset"]) else None)[3::4]
    qs["S_glob"] = leaky_surprise(go, float(p_glo["param"]), gr if bool(p_glo["reset"]) else None)
    return qs[["seq_idx", "S_loc", "S_glob"]]


# ---------------------------------------------------------------- random-effects BMS
def rfx_bms(L, n_samples=100000, seed=SEED):
    '''Random-effects Bayesian model selection (Stephan et al., 2009) with protected exceedance
    probabilities (Rigoux et al., 2014). L: subjects x models log evidence (here: held-out log-likelihood).
    Variational update of the Dirichlet posterior over model frequencies; exceedance probabilities by
    sampling; Bayesian omnibus risk (BOR) = posterior probability that all models are equally frequent.'''
    n, K = L.shape
    a0 = np.ones(K); a = a0.copy()
    for _ in range(1000):
        lu = L + digamma(a) - digamma(a.sum())
        g = np.exp(lu - logsumexp(lu, axis=1, keepdims=True))   # posterior model assignment per subject
        a_new = a0 + g.sum(0)
        if np.max(np.abs(a_new - a)) < 1e-8:
            a = a_new; break
        a = a_new
    r = np.random.default_rng(seed).dirichlet(a, n_samples)
    ep = np.bincount(r.argmax(1), minlength=K) / n_samples      # exceedance probability
    F0 = np.sum(logsumexp(L, axis=1) - np.log(K))                # evidence for the null (equal frequencies)
    Elogr = digamma(a) - digamma(a.sum())
    kl = gammaln(a.sum()) - gammaln(a).sum() - gammaln(a0.sum()) + gammaln(a0).sum() + np.sum((a - a0) * Elogr)
    F1 = np.sum(g * (L + Elogr - np.log(g + 1e-300))) - kl       # evidence for the RFX model
    bor = 1 / (1 + np.exp(np.clip(F1 - F0, -700, 700)))
    return dict(alpha=a, expected_freq=a / a.sum(), ep=ep, bor=bor, pxp=ep * (1 - bor) + bor / K)


def jzs_bf10(d, r=np.sqrt(2) / 2):
    '''JZS Bayes factor for a one-sample/paired t-test (Rouder et al., 2009), as 05_reset_validation.py.'''
    d = np.asarray(d, float); n = len(d); v = n - 1
    t = d.mean() / (d.std(ddof=1) / np.sqrt(n))
    f = lambda g: ((1 + n * g * r ** 2) ** -0.5 * (1 + t ** 2 / ((1 + n * g * r ** 2) * v)) ** (-(v + 1) / 2)
                   * (2 * np.pi) ** -0.5 * g ** -1.5 * np.exp(-1 / (2 * g)))
    return integrate.quad(f, 0, np.inf)[0] / (1 + t ** 2 / v) ** (-(v + 1) / 2)


REPORT = []      # (section, quantity, value recomputed here) -> compared with the Results draft in Section 5
def report(section, name, value):
    REPORT.append(dict(section=section, quantity=name, notebook=value))
    return value

## 1. Figure 1 — local deviance gives an early frontocentral MMN, global deviance a late parietal P3b

**Logic.** Each quintlet has two orthogonal labels: *local* (5th sound repeats, LS, or changes, LD) and *global*
(the quintlet is the block's standard, GS, or its rare deviant, GD). Physically identical quintlets are standards in
one block and deviants in another, so GD − GS is acoustically matched. Inference is on the a-priori ROI × window
means; cluster permutation tests describe where/when effects occur without a-priori windows.

In [ ]:
times, chs, D, subs = load_erps()
n = len(subs)
print(f"{n} participants, {len(chs)} channels, {len(times)} samples ({times[0]:.2f} to {times[-1]:.2f} s)")

fig1 = {}
for key, r in ROI.items():
    std = roi_wave(D, r["std"], chs, r["chs"]); dev = roi_wave(D, r["dev"], chs, r["chs"])
    diff = dev - std
    res, per_sub = window_test(diff, times, r["win"])
    # peak of the group-mean difference: most extreme point from 50 ms before the window to its end
    sel = (times >= r["win"][0] - 0.05) & (times <= r["win"][1])
    res["peak_ms"] = times[sel][np.argmax(np.abs(diff.mean(0)[sel]))] * 1e3
    res["clusters"] = cluster_1d(diff, times)
    fig1[key] = res
    print(f"\n{key.upper()}  {r['dev']} - {r['std']}, {int(r['win'][0]*1e3)}-{int(r['win'][1]*1e3)} ms, ROI {r['chs']}")
    print(f"  mean {res['mean']:+.2f} µV (95% CI ± {res['ci95']:.2f}), t({n-1}) = {res['t']:.2f}, p = {res['p']:.2g}, "
          f"{res['same_sign']}/{n} same sign, peak {res['peak_ms']:.0f} ms")
    print("  temporal clusters (0-700 ms):"); print(res["clusters"].round(4).to_string(index=False))

report(1, "MMN mean (µV)", fig1["local"]["mean"]); report(1, "MMN t", fig1["local"]["t"])
report(1, "MMN 95% CI half-width", fig1["local"]["ci95"]); report(1, "MMN peak (ms)", fig1["local"]["peak_ms"])
report(1, "P3b mean (µV)", fig1["global"]["mean"]); report(1, "P3b t", fig1["global"]["t"])
report(1, "P3b 95% CI half-width", fig1["global"]["ci95"]); report(1, "P3b peak (ms)", fig1["global"]["peak_ms"])

### 1.1 Spatio-temporal clusters (Figure 1C)

Same data, all 62 channels × 0–700 ms; channel neighbourhoods from the montage; sign-flip permutation.
Cluster onsets are *descriptive*: a cluster test controls the family-wise error of "an effect exists in this
cluster", not of its exact extent (Sassenhagen & Draschkow, 2019).

In [ ]:
info = mne.io.read_info(ERP_DIR / "info.fif")
adjacency, _ = mne.channels.find_ch_adjacency(info, "eeg")
sel = (times >= 0) & (times <= 0.7); ts = times[sel]
st_clusters = {}
for key, (dev, std) in {"local": ("LD", "LS"), "global": ("GD", "GS")}.items():
    X = (D[dev] - D[std])[:, :, sel].transpose(0, 2, 1)          # subjects x time x channels
    T, clusters, pv, _ = mne.stats.spatio_temporal_cluster_1samp_test(
        X, threshold=stats.t.ppf(0.975, n - 1), n_permutations=N_PERM, adjacency=adjacency,
        tail=0, seed=SEED, out_type="mask", verbose=False)
    rows = []
    for c, p in zip(clusters, pv):
        if p < 0.05:
            tt = np.where(c.any(1))[0]
            rows.append(dict(sign="+" if T[c].mean() > 0 else "-", p=p, onset_ms=ts[tt[0]] * 1e3,
                             offset_ms=ts[tt[-1]] * 1e3, n_channels=int(c.any(0).sum()), peak_abs_t=np.abs(T[c]).max()))
    st_clusters[key] = pd.DataFrame(rows).sort_values("onset_ms")
    print(f"\n{key}: significant spatio-temporal clusters"); print(st_clusters[key].round(4).to_string(index=False))

### 1.2 Local × global: 2 × 2 repeated-measures ANOVA at each a-priori window

The four 2 × 2 cells are averaged in each ROI × window per participant (`sub-*_cond4.tsv`, written by 07 from
the same epochs). Simple effects are paired t-tests. Section 4 checks the ANOVA against the equivalent contrast
t-tests (for a 2 × 2 within design, each F(1, n−1) equals the squared t of its contrast).

In [ ]:
c4 = pd.concat([pd.read_csv(f, sep="\t") for f in sorted(ERP_DIR.glob("sub-*_cond4.tsv"))])
cells = c4[c4["source"] == "data"].copy()
cells["local"], cells["global"] = cells["condition"].str[:2], cells["condition"].str[2:]
anova = {}
for comp in ["local", "global"]:
    q = cells[cells["component"] == comp]
    aov = pg.rm_anova(data=q, dv="amp", within=["local", "global"], subject="subject", detailed=True)
    piv = q.pivot(index="subject", columns="condition", values="amp")[CONDS]
    anova[comp] = (aov, piv)
    print(f"\n{comp} window — cell means (µV):", piv.mean().round(2).to_dict())
    print(aov[["Source", "F", "ddof1", "ddof2", [c for c in aov.columns if c.startswith("p")][0], "ng2"]].round(4).to_string(index=False))
    for name, a, b in [("LD: GD vs GS", "LDGD", "LDGS"), ("LS: GD vs GS", "LSGD", "LSGS"),
                       ("GS: LD vs LS", "LDGS", "LSGS"), ("GD: LD vs LS", "LDGD", "LSGD")]:
        d = piv[a] - piv[b]; t = stats.ttest_rel(piv[a], piv[b])
        print(f"   {name}: {d.mean():+.2f} µV, t({n-1}) = {t.statistic:.2f}, p = {t.pvalue:.2g}, "
              f"{int((np.sign(d) == np.sign(d.mean())).sum())}/{n}")
        report(1, f"{comp} simple effect {name} t", t.statistic)

pcol = lambda a: [c for c in a.columns if c.startswith("p")][0]
for comp in ["local", "global"]:
    aov = anova[comp][0].set_index("Source")
    for eff in ["local", "global", "local * global"]:
        report(1, f"ANOVA {comp}-window {eff} F", float(aov.loc[eff, "F"]))

### 1.3 Reference control

Re-referencing is linear, so it is applied to the averages: linked mastoids = subtract the mean of TP9/TP10.
A supratemporal MMN generator predicts a larger frontocentral MMN under this reference and a polarity
inversion at the mastoids under the average reference.

In [ ]:
mi, fz = [chs.index(c) for c in ["TP9", "TP10"]], chs.index("Fz")
w = (times >= 0.08) & (times <= 0.14)
avg_fz = (D["LD"][:, fz] - D["LS"][:, fz])[:, w].mean(1)
lm = {c: D[c] - D[c][:, mi, :].mean(1, keepdims=True) for c in ["LD", "LS"]}
lm_fz = (lm["LD"][:, fz] - lm["LS"][:, fz])[:, w].mean(1)
mast = ((D["LD"] - D["LS"])[:, mi].mean(1))[:, w].mean(1)
for name, x in [("Fz, average reference", avg_fz), ("Fz, linked mastoids", lm_fz), ("TP9/TP10, average reference", mast)]:
    t = stats.ttest_1samp(x, 0)
    print(f"{name:30s} {x.mean():+.2f} µV, t({n-1}) = {t.statistic:.2f}, p = {t.pvalue:.2g}, {int((np.sign(x)==np.sign(x.mean())).sum())}/{n}")
report(1, "MMN Fz linked mastoids (µV)", lm_fz.mean()); report(1, "Mastoid inversion (µV)", mast.mean())

In [ ]:
# ---------------------------------------------------------------- figures: shared settings
from IPython.display import Image, display
import sys
PROJECT_ROOT = ROOT                      # every figure function reads its inputs relative to this
(PROJECT_ROOT / "figures").mkdir(exist_ok=True)
FIG_LOG = []                             # figures made in this session (section 7)

def run_figure(name, fn, *args, **kwargs):
    '''Run one figure function; report (rather than stop the notebook on) a missing input.'''
    t0 = time.time()
    try:
        fn(*args, **kwargs)
        FIG_LOG.append((name, "ok", round(time.time() - t0, 1)))
    except Exception as e:                # e.g. template head or DCM fits not available on this computer
        FIG_LOG.append((name, f"FAILED: {type(e).__name__}: {e}", round(time.time() - t0, 1)))
        print(f"{name} failed: {type(e).__name__}: {e}")

### Figure 1 — design, topographies, ERPs and participant effects

`scripts/14_figure1.py`. From the script's header:

```text
14_figure1.py — Figure 1: task, contrasts, topographies, MMN and P3b.

  A  Task: quintlet structure and timing; example blocks for the two rules within a sound pair
  B  The 2 x 2 design and the two contrasts (local: LD − LS; global: GD − GS) and what they index
  C  Group difference topographies 0–700 ms (50-ms bins) for both contrasts; rings = ROI electrodes,
     dots = significant spatio-temporal cluster
  D  MMN: frontocentral ROI, LS vs LD and difference; a-priori window shaded; bar = significant
     temporal cluster of the difference
  E  P3b: parietal ROI, GS vs GD and difference; same conventions

Input: per-subject condition averages cached by 07_header1_erps.py (derivatives/header1_erps/<sub>.npz;
average reference, 30 Hz low-pass, baseline −100–0 ms) and info.fif.
Output: figures/Figure1.png and figures/Figure1.pdf; derivatives/header1_erps/figure1_stats.json

Usage:  python scripts/14_figure1.py            # full version  -> Figure1.png/.pdf
        python scripts/14_figure1.py --lean     # 100-ms maps, no cluster dots, methods text moved to legend
                                                 #          -> Figure1_lean.png/.pdf
        python scripts/14_figure1.py --erp      # panels D & E only, editable text -> Figure1_DE.pdf/.svg/.png
```

In [ ]:
# ---- 14_figure1.py (inline; rebuild the notebook after editing the script)
def make_figure1(lean=False, erp_only=False):
    from pathlib import Path
    import json
    import sys

    import numpy as np
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle, FancyBboxPatch, FancyArrowPatch
    import seaborn as sns
    import mne
    from scipy import stats

    # ----------------------------------------------------------------------------- CONFIG
    ROOT = PROJECT_ROOT
    CACHE_DIR = ROOT / "derivatives" / "header1_erps"
    FIG_DIR = ROOT / "figures"
    TONE = {"A": "#4C72B0", "B": "#DD8452"}
    C_LOC, C_GLOB, C_STD = "#C44E52", "#8172B3", "0.35"
    ROI = {"local": dict(chs=["Fz", "FC1", "FC2", "F1", "F2", "Cz"], win=(0.08, 0.14), dev="LD", std="LS",
                         ring="#E07B00"),
           "global": dict(chs=["Pz", "P1", "P2", "CPz", "POz", "CP1", "CP2"], win=(0.30, 0.70), dev="GD", std="GS",
                          ring="#1B9E77")}
    LEAN = lean
    ERP_ONLY = erp_only          # panels D & E alone, as a vector file with editable text
    # Editable text in vector output: TrueType (Type 42) fonts in PDF, real <text> elements in SVG.
    TEXT_RC = {"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
               "font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "Helvetica", "DejaVu Sans"]}
    _b = np.round(np.arange(0.0, 0.701, 0.05), 3)
    # lean: four windows matched to the effects (MMN, early positivity, early/late P3b) instead of uniform bins
    TOPO_WINS = ([(0.08, 0.14), (0.16, 0.30), (0.30, 0.50), (0.50, 0.70)] if LEAN else list(zip(_b[:-1], _b[1:])))
    N_PERM = 2000
    SEED = 42
    # ------------------------------------------------------------------------------------

    mne.set_log_level("ERROR")


    # ============================================================ drawing helpers
    def quintlet(ax, x, y, q, w=0.012, gap=0.004, h=0.07, edge=None, lw=1.6):
        """Draw a quintlet (string of 'A'/'B') as five bars; B bars taller. Returns right edge."""
        x0 = x
        for c in q:
            hh = h if c == "A" else h * 1.35
            ax.add_patch(Rectangle((x, y), w, hh, fc=TONE[c], ec="none"))
            x += w + gap
        if edge is not None:
            ax.add_patch(FancyBboxPatch((x0 - 0.004, y - 0.012), x - x0 + 0.004, h * 1.35 + 0.024,
                                        boxstyle="round,pad=0.002,rounding_size=0.006", fc="none", ec=edge, lw=lw))
        return x


    def panel_label(ax, s, x=-0.02, y=1.02):
        ax.text(x, y, s, transform=ax.transAxes, fontsize=16, weight="bold", va="bottom", ha="left")


    def draw_task(ax):
        ax.set_axis_off()
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        # --- one quintlet, with timing
        ax.text(0.0, 0.93, "One trial = a quintlet of 50-ms chords", fontsize=10.5, weight="bold", va="top")
        x0, y0, w, step = 0.03, 0.60, 0.035, 0.075
        for k, c in enumerate("AAAAB"):
            ax.add_patch(Rectangle((x0 + k * step, y0), w, 0.11 if c == "A" else 0.15, fc=TONE[c], ec="k" if k == 4 else "none",
                                   lw=1.5))
        ax.annotate("", xy=(x0 + step, y0 - 0.03), xytext=(x0, y0 - 0.03),
                    arrowprops=dict(arrowstyle="<->", lw=0.9, color="0.3"))
        ax.text(x0 + step / 2, y0 - 0.045, "SOA ≈ 157 ms", ha="center", va="top", fontsize=8)
        ax.text(x0 + 4 * step + w / 2, y0 + 0.16, "5th sound (t = 0)", ha="center", va="bottom", fontsize=8.5)
        ax.annotate("", xy=(x0 + 4 * step + w + 0.13, y0 + 0.05), xytext=(x0 + 4 * step + w + 0.01, y0 + 0.05),
                    arrowprops=dict(arrowstyle="<->", lw=0.9, color="0.3"))
        ax.text(x0 + 4 * step + w + 0.07, y0 + 0.07, "800–1100 ms", ha="center", va="bottom", fontsize=8)
        ax.text(0.55, 0.92, "■", color=TONE["A"], fontsize=11, va="top")
        ax.text(0.58, 0.92, "chord A" if LEAN else "chord A  350/700/1400 Hz", fontsize=8.5, va="top")
        ax.text(0.55, 0.85, "■", color=TONE["B"], fontsize=11, va="top")
        ax.text(0.58, 0.85, "chord B" if LEAN else "chord B  500/1000/2000 Hz", fontsize=8.5, va="top")
        ax.text(0.55, 0.76, "Task: silently count global deviants" if LEAN else
                "Task: silently count global deviants\n(totals reported after every 4 blocks)",
                fontsize=8.5, va="top", style="italic")
        # --- two example blocks (same sound pair, opposite rules)
        ax.text(0.0, 0.52, "Blocks: the standard quintlet (80%) sets the rule; the other is the deviant (20%)",
                fontsize=10.5, weight="bold", va="top")
        rows = [("Repeat-rule block\n(standard AAAAA)", ["AAAAA"] * 4 + ["AAAAB"] + ["AAAAA"] * 3 + ["AAAAB"] + ["AAAAA"] * 2, "AAAAB"),
                ("Change-rule block\n(standard AAAAB)", ["AAAAB"] * 4 + ["AAAAA"] + ["AAAAB"] * 3 + ["AAAAA"] + ["AAAAB"] * 2, "AAAAA")]
        for r, (lab, seq, dev) in enumerate(rows):
            y = 0.30 - r * 0.22
            ax.text(0.0, y + 0.045, lab, fontsize=8.5, va="center")
            x = 0.20
            for q in seq:
                x = quintlet(ax, x, y, q, w=0.009, gap=0.0025, h=0.06, edge=C_GLOB if q == dev else None) + 0.010
            ax.text(x, y + 0.04, "…", fontsize=12, va="center")
        if LEAN:
            return
        ax.text(0.20, -0.02, "8 blocks × 120–164 quintlets (AAAAA, AAAAB, BBBBB, BBBBA standards, 2 blocks each)\n"
                "≥ 10 standards open every block; 5-s silence between blocks;  purple box = global deviant",
                fontsize=7.5, va="bottom", color="0.3")


    def draw_contrasts(ax):
        ax.set_axis_off()
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        cx = [0.33, 0.62]
        cy = [0.72, 0.46]
        ax.text(cx[0], 0.94, "Local standard (LS)\n5th = 4th", ha="center", va="top", fontsize=9, weight="bold")
        ax.text(cx[1], 0.94, "Local deviant (LD)\n5th ≠ 4th", ha="center", va="top", fontsize=9, weight="bold",
                color=C_LOC)
        ax.text(0.10, cy[0] + 0.03, "Global\nstandard\n(GS)", ha="center", va="center", fontsize=9, weight="bold")
        ax.text(0.10, cy[1] + 0.03, "Global\ndeviant\n(GD)", ha="center", va="center", fontsize=9, weight="bold",
                color=C_GLOB)
        cells = {(0, 0): ("AAAAA", "in AAAAA block"), (0, 1): ("AAAAB", "in AAAAB block"),
                 (1, 0): ("AAAAA", "in AAAAB block"), (1, 1): ("AAAAB", "in AAAAA block")}
        for (r, c), (q, ctx) in cells.items():
            ax.add_patch(FancyBboxPatch((cx[c] - 0.12, cy[r] - 0.09), 0.24, 0.23,
                                        boxstyle="round,pad=0.005,rounding_size=0.015", fc="0.97",
                                        ec=C_GLOB if r == 1 else "0.7", lw=1.4 if r == 1 else 0.8))
            quintlet(ax, cx[c] - 0.085, cy[r] + 0.01, q, w=0.024, gap=0.009, h=0.07,
                     edge=C_LOC if c == 1 else None, lw=1.2)
            ax.text(cx[c], cy[r] - 0.055, f"{['LS', 'LD'][c]}/{['GS', 'GD'][r]}  ·  {ctx}", ha="center",
                    va="center", fontsize=7.5)
        # contrast explanations
        ax.text(0.02, 0.335, "Local effect = LD − LS", color=C_LOC, fontsize=10, weight="bold", va="top")
        ax.text(0.02, 0.29, "a change at the 5th sound (tone-to-tone surprise)\n→ early frontocentral MMN, auditory cortex",
                fontsize=8.5, va="top")
        ax.add_patch(FancyArrowPatch((0.905, cy[0] - 0.02), (0.905, cy[1] + 0.10), arrowstyle="-|>",
                                     mutation_scale=12, color=C_GLOB, lw=1.5))
        ax.text(0.02, 0.205, "Global effect = GD − GS", color=C_GLOB, fontsize=10, weight="bold", va="top")
        ax.text(0.02, 0.16, "the quintlet breaks the block's rule\n→ late parietal P3b", fontsize=8.5, va="top")
        if LEAN:
            return
        ax.text(0.02, -0.04, "Physically identical quintlets swap roles across blocks (e.g. AAAAB is GS in one\n"
                "block and GD in another), so the global effect is not a stimulus difference.\nSame design for the B pair.",
                ha="left", va="bottom", fontsize=7.5, color="0.3")


    # ============================================================ data
    def load():
        files = sorted(CACHE_DIR.glob("sub-*.npz"))
        z0 = np.load(files[0])
        info = mne.io.read_info(CACHE_DIR / "info.fif")
        D = {c: np.stack([np.load(f)[f"topo__{c}"] for f in files]) for c in ["LS", "LD", "GS", "GD"]}
        return z0["times"], list(z0["ch_names"]), info, D, len(files)


    def roi_wave(D, c, chs, roi_chs):
        return D[c][:, [chs.index(x) for x in roi_chs]].mean(1)          # subj x time


    def cluster_1d(diff, times, tmin=0.0, tmax=0.7):
        sel = (times >= tmin) & (times <= tmax)
        n = len(diff)
        T, clusters, pv, _ = mne.stats.permutation_cluster_1samp_test(
            diff[:, sel], threshold=stats.t.ppf(0.975, n - 1), n_permutations=N_PERM, tail=0, seed=SEED,
            out_type="indices", verbose=False)
        sig = np.zeros(len(times), bool)
        cl = []
        idx = np.where(sel)[0]
        for c, p in zip(clusters, pv):
            if p < 0.05:
                sig[idx[c[0]]] = True
                cl.append(dict(start_ms=float(times[idx[c[0][0]]] * 1e3), end_ms=float(times[idx[c[0][-1]]] * 1e3),
                               p=float(p), sign="+" if T[c[0]].mean() > 0 else "−"))
        return sig, cl


    def erp_panel(ax, times, D, chs, key, n, label_std, label_dev, col_dev, title):
        r = ROI[key]
        s, d = roi_wave(D, r["std"], chs, r["chs"]), roi_wave(D, r["dev"], chs, r["chs"])
        diff = d - s
        t = times * 1e3
        ax.axvspan(r["win"][0] * 1e3, r["win"][1] * 1e3, color="0.92", zorder=0)
        for x, col, lab, ls, lw in [(s, C_STD, label_std, "-", 1.8), (d, col_dev, label_dev, "-", 1.8),
                                    (diff, "k", f"{r['dev']} − {r['std']}", "--", 1.3)]:
            m, se = x.mean(0), x.std(0, ddof=1) / np.sqrt(n)
            ax.plot(t, m, color=col, lw=lw, ls=ls, label=lab)
            ax.fill_between(t, m - se, m + se, color=col, alpha=0.18, lw=0)
        sig, cl = cluster_1d(diff, times)
        ax.plot(np.where(sig, t, np.nan), np.full(len(t), 0.03), color="k", lw=4, solid_capstyle="butt",
                transform=ax.get_xaxis_transform())
        for k in range(1, 5):
            ax.axvline(-157 * k, color="0.8", lw=0.6, ls=":")
        ax.axvline(0, color="k", lw=0.9)
        ax.axhline(0, color="k", lw=0.5)
        w = (times >= r["win"][0]) & (times <= r["win"][1])
        mw = diff[:, w].mean(1)
        tt = stats.ttest_1samp(mw, 0)
        pk = times[np.argmax(np.abs(diff.mean(0)) * ((times >= r["win"][0] - 0.05) & (times <= r["win"][1])))] * 1e3
        ax.set(xlim=(-200, 700), xlabel="Time re 5th sound (ms)", ylabel="µV")
        ax.set_title(f"{title}\n{', '.join(r['chs'])}", fontsize=10)
        ax.text(0.99, 0.97, f"{int(r['win'][0] * 1e3)}–{int(r['win'][1] * 1e3)} ms: {mw.mean():+.2f} µV\n"
                f"t({n - 1}) = {tt.statistic:.1f}, p = {tt.pvalue:.1g}\n{int((np.sign(mw) == np.sign(mw.mean())).sum())}/{n} subjects",
                transform=ax.transAxes, ha="right", va="top", fontsize=8.5)
        ax.legend(frameon=False, fontsize=8.5, loc="upper left")
        return dict(window_ms=[int(r["win"][0] * 1e3), int(r["win"][1] * 1e3)], mean_diff_uV=float(mw.mean()),
                    t=float(tt.statistic), p=float(tt.pvalue), peak_ms=float(pk),
                    n_same_sign=int((np.sign(mw) == np.sign(mw.mean())).sum()), temporal_clusters=cl)


    def topo_rows(fig, gs_cell, times, chs, info, D, n):
        adjacency, _ = mne.channels.find_ch_adjacency(info, "eeg")
        from mne.channels.layout import _find_topomap_coords
        pos = _find_topomap_coords(info, picks=mne.pick_types(info, eeg=True))
        nb = len(TOPO_WINS)
        sub = gs_cell.subgridspec(2, nb + 1, width_ratios=[1] * nb + [0.07], hspace=0.08, wspace=0.05)
        out = {}
        for r, key in enumerate(["local", "global"]):
            roi = ROI[key]
            diff = D[roi["dev"]] - D[roi["std"]]
            sel = (times >= 0) & (times <= 0.7)
            T, clusters, pv, _ = mne.stats.spatio_temporal_cluster_1samp_test(
                diff[:, :, sel].transpose(0, 2, 1), threshold=stats.t.ppf(0.975, n - 1), n_permutations=N_PERM,
                adjacency=adjacency, tail=0, seed=SEED, out_type="mask", verbose=False)
            mask = np.zeros_like(T, bool)
            for c, p in zip(clusters, pv):
                if p < 0.05:
                    mask |= c
            out[key] = dict(cluster_p=[float(p) for p in sorted(pv)[:3]])
            M = diff.mean(0)[:, sel]
            ts = times[sel]
            lim = np.ceil(np.abs(M).max() * 10) / 10
            ri = [chs.index(c) for c in roi["chs"]]
            for k in range(nb):
                ax = fig.add_subplot(sub[r, k])
                b = (ts >= TOPO_WINS[k][0]) & (ts < TOPO_WINS[k][1] + (1e-9 if k == nb - 1 else 0))
                im, _ = mne.viz.plot_topomap(M[:, b].mean(1), info, axes=ax, show=False, cmap="RdBu_r", vlim=(-lim, lim),
                                             contours=0, sensors=False, mask=None if LEAN else mask[b].any(0),
                                             mask_params=dict(marker="o", markerfacecolor="k", markeredgecolor="k",
                                                              markersize=1.8))
                ax.scatter(pos[ri, 0], pos[ri, 1], s=40 if LEAN else 26, facecolor="none", edgecolor=roi["ring"], lw=1.2, zorder=5)
                if TOPO_WINS[k][0] < roi["win"][1] - 1e-9 and TOPO_WINS[k][1] > roi["win"][0] + 1e-9:
                    ax.plot([0.1, 0.9], [-0.04, -0.04], transform=ax.transAxes, color=roi["ring"], lw=3.5, clip_on=False,
                            solid_capstyle="butt")
                if r == 0:
                    ax.set_title(f"{round(TOPO_WINS[k][0] * 1e3)}–{round(TOPO_WINS[k][1] * 1e3)}" + (" ms" if LEAN else ""), fontsize=9 if LEAN else 8)
                if k == 0:
                    ax.text(-0.25, 0.5, ["Local\nLD − LS", "Global\nGD − GS"][r], transform=ax.transAxes, rotation=90,
                            ha="center", va="center", fontsize=9.5, weight="bold", color=[C_LOC, C_GLOB][r])
            cb = fig.colorbar(im, cax=fig.add_subplot(sub[r, -1]))
            cb.set_label("µV", fontsize=8)
            cb.ax.tick_params(labelsize=7)
        return out


    # ============================================================ print-size ERP block (--erp)
    PRINT_MM = (165.467, 86.011)          # target box, width x height


    def erp_print_block(times, D, chs, n):
        """D/E/F at final print size: conditions (top) and difference (bottom) for local and global,
        sharing x; right column = per-participant window means. Editable text, 7-pt type."""
        mm = 1 / 25.4
        plt.rcParams.update({"font.size": 7, "axes.titlesize": 7, "axes.labelsize": 7, "xtick.labelsize": 6,
                             "ytick.labelsize": 6, "legend.fontsize": 6, "axes.linewidth": 0.6,
                             "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.major.size": 2.5,
                             "ytick.major.size": 2.5, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5,
                             "axes.labelpad": 2})
        fig = plt.figure(figsize=(PRINT_MM[0] * mm, PRINT_MM[1] * mm))
        gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.5], height_ratios=[1.15, 1],
                              left=0.065, right=0.99, bottom=0.12, top=0.90, wspace=0.36, hspace=0.12)
        t = times * 1e3
        out = {}

        def trace(ax, X, col, lab, ls="-", lw=1.0):
            m, se = X.mean(0), X.std(0, ddof=1) / np.sqrt(n)
            ax.plot(t, m, color=col, lw=lw, ls=ls, label=lab)
            ax.fill_between(t, m - se, m + se, color=col, alpha=0.2, lw=0)

        def furniture(ax, win):
            ax.axvspan(win[0] * 1e3, win[1] * 1e3, color="0.92", zorder=0, lw=0)
            ax.axvline(0, color="k", lw=0.6)
            ax.axhline(0, color="k", lw=0.4)
            ax.set_xlim(-200, 700)
            ax.set_xticks([-200, 0, 200, 400, 600])

        for c, (key, col, lab, names, title) in enumerate([
                ("local", C_LOC, "D", ("LS", "LD"), "Local: MMN (frontocentral)"),
                ("global", C_GLOB, "E", ("GS", "GD"), "Global: P3b (parietal)")]):
            r = ROI[key]
            s_, d_ = roi_wave(D, r["std"], chs, r["chs"]), roi_wave(D, r["dev"], chs, r["chs"])
            diff = d_ - s_
            axc = fig.add_subplot(gs[0, c])
            axd = fig.add_subplot(gs[1, c], sharex=axc)
            for ax in (axc, axd):
                furniture(ax, r["win"])
            trace(axc, s_, C_STD, f"standard ({names[0]})")
            trace(axc, d_, col, f"deviant ({names[1]})")
            trace(axd, diff, "k", f"{names[1]} − {names[0]}")
            sig, cl = cluster_1d(diff, times)
            axd.plot(np.where(sig, t, np.nan), np.full(len(t), 0.04), color=col, lw=2.2, solid_capstyle="butt",
                     transform=axd.get_xaxis_transform())
            axc.set_title(title, fontsize=7, pad=3)
            axc.legend(frameon=False, loc="upper left", handlelength=1.4, borderaxespad=0.2, labelspacing=0.25)
            axd.legend(frameon=False, loc="upper left", handlelength=1.4, borderaxespad=0.2)
            plt.setp(axc.get_xticklabels(), visible=False)
            axc.set_ylabel("µV")
            axd.set_ylabel("Δ µV")
            axd.set_xlabel("Time from 5th sound (ms)")
            sns.despine(ax=axc)
            sns.despine(ax=axd)
            axc.text(-0.2, 1.08, lab, transform=axc.transAxes, fontsize=9, weight="bold", va="bottom")
            w = (times >= r["win"][0]) & (times <= r["win"][1])
            out[key] = dict(per_subject=diff[:, w].mean(1), win=r["win"], col=col, names=names)
        fig.align_ylabels()

        # ---- F: per-participant effects
        axf = fig.add_subplot(gs[:, 2])
        rng = np.random.default_rng(1)
        for k, key in enumerate(["local", "global"]):
            o = out[key]
            y = o["per_subject"]
            tt = stats.ttest_1samp(y, 0)
            ci = stats.t.ppf(0.975, n - 1) * y.std(ddof=1) / np.sqrt(n)
            x = k + rng.uniform(-0.12, 0.12, n)
            axf.scatter(x, y, s=9, color=o["col"], alpha=0.75, lw=0, zorder=3)
            axf.errorbar(k + 0.3, y.mean(), yerr=ci, fmt="o", color="k", ms=3.2, lw=0.9, capsize=0, zorder=4)
            same = int((np.sign(y) == np.sign(y.mean())).sum())
            o.update(t=float(tt.statistic), p=float(tt.pvalue), mean=float(y.mean()), ci=float(ci), n_same=same)
        axf.set_xticks([0, 1])
        axf.set_xticklabels([f"MMN\n{int(ROI['local']['win'][0]*1e3)}–{int(ROI['local']['win'][1]*1e3)}",
                             f"P3b\n{int(ROI['global']['win'][0]*1e3)}–{int(ROI['global']['win'][1]*1e3)}"])
        for k, key in enumerate(["local", "global"]):
            o = out[key]
            axf.get_xticklabels()[k].set_color(o["col"])
            axf.text(k + 0.1, 1.0, f"t = {o['t']:.1f}".replace("-", "−") + f"\n{o['n_same']}/{n}", transform=axf.get_xaxis_transform(),
                     ha="center", va="bottom", fontsize=6, linespacing=1.1)
        axf.axhline(0, color="k", lw=0.4)
        axf.set_xlim(-0.4, 1.55)
        axf.set_ylabel("Deviant − standard (µV)")
        axf.set_xlabel("Window (ms)")
        sns.despine(ax=axf)
        axf.text(-0.45, 1.08, "F", transform=axf.transAxes, fontsize=9, weight="bold", va="bottom")

        FIG_DIR.mkdir(exist_ok=True)
        fig.savefig(FIG_DIR / "Figure1_DE.pdf")
        fig.savefig(FIG_DIR / "Figure1_DE.svg")
        svg = FIG_DIR / "Figure1_DE.svg"          # SVG names fonts: point it at Arial (Liberation Sans = same metrics)
        svg.write_text(svg.read_text().replace("Liberation Sans", "Arial"))
        fig.savefig(FIG_DIR / "Figure1_DE.png", dpi=400)
        w_in, h_in = fig.get_size_inches()
        print(f"Figure -> {FIG_DIR / 'Figure1_DE'}.pdf/.svg/.png  ({w_in * 25.4:.1f} x {h_in * 25.4:.1f} mm)")
        for key in ("local", "global"):
            o = out[key]
            print(key, f"mean {o['mean']:+.2f} µV, 95% CI ±{o['ci']:.2f}, t = {o['t']:.2f}, p = {o['p']:.2g}, {o['n_same']}/{n}")


    def main():
        times, chs, info, D, n = load()
        sns.set_theme(context="notebook", style="ticks", rc=TEXT_RC)
        if ERP_ONLY:
            erp_print_block(times, D, chs, n)
            return
        fig = plt.figure(figsize=(16, 17))
        outer = fig.add_gridspec(3, 1, height_ratios=[1.15, 0.8 if LEAN else 0.95, 1.05], hspace=0.28)
        top = outer[0].subgridspec(1, 2, width_ratios=[1.45, 1], wspace=0.08)
        axA = fig.add_subplot(top[0, 0])
        draw_task(axA)
        panel_label(axA, "A", x=-0.01, y=0.97)
        axB = fig.add_subplot(top[0, 1])
        draw_contrasts(axB)
        panel_label(axB, "B", x=-0.02, y=0.97)

        topo_stats = topo_rows(fig, outer[1], times, chs, info, D, n)
        fig.text(0.115, outer[1].get_position(fig).y1 + (0.02 if LEAN else 0.005), "C", fontsize=16, weight="bold", va="bottom")
        fig.text(0.14, outer[1].get_position(fig).y1 + (0.022 if LEAN else 0.007),
                 ("Difference topographies after the 5th sound (rings = ROI; bar = analysis window)" if LEAN else
                  "Difference topographies, 0–700 ms after the 5th sound (50-ms bins; rings = ROI; coloured bar = "
                  "analysis window; dots = significant spatio-temporal cluster)"), fontsize=9.5, va="bottom")

        bot = outer[2].subgridspec(1, 2, wspace=0.22)
        axD = fig.add_subplot(bot[0, 0])
        sD = erp_panel(axD, times, D, chs, "local", n, "local standard (LS)", "local deviant (LD)", C_LOC,
                       "Local effect — MMN (frontocentral ROI)")
        panel_label(axD, "D", x=-0.1, y=1.06)
        axE = fig.add_subplot(bot[0, 1])
        sE = erp_panel(axE, times, D, chs, "global", n, "global standard (GS)", "global deviant (GD)", C_GLOB,
                       "Global effect — P3b (parietal ROI)")
        panel_label(axE, "E", x=-0.1, y=1.06)
        for ax in (axD, axE):
            sns.despine(ax=ax)
        FIG_DIR.mkdir(exist_ok=True)
        stem = "Figure1_lean" if LEAN else "Figure1"
        fig.savefig(FIG_DIR / f"{stem}.png", dpi=200, bbox_inches="tight")
        fig.savefig(FIG_DIR / f"{stem}.pdf", bbox_inches="tight")
        if LEAN:
            print(f"Figure -> {FIG_DIR / stem}.png")
            return
        st = dict(n_subjects=n, MMN=sD, P3b=sE, topography_cluster_p=topo_stats)
        (CACHE_DIR / "figure1_stats.json").write_text(json.dumps(st, indent=2))
        print(json.dumps(st, indent=2))
        print(f"Figure -> {FIG_DIR / 'Figure1.png'}")

    return main()


run_figure('Figure 1', make_figure1)                         # full figure (A–F)
run_figure('Figure 1 lean', make_figure1, lean=True)         # lean version (fewer topography bins)
run_figure('Figure 1 D–F print', make_figure1, erp_only=True)  # D–F at print size, editable text

plt.close('all')
for f in ['Figure1.png', 'Figure1_lean.png', 'Figure1_DE.png']:
    p = PROJECT_ROOT / 'figures' / f
    display(Image(filename=str(p), width=900) if p.exists() else f'missing: {p}')

## 2. Figure 2 — each response follows a different learner

**Models** of the single-trial window amplitude (all with intercept + linear drift over the session):

| | regressor(s) | free parameters (chosen inside CV) |
|---|---|---|
| M0 categories | LD, GD, LD×GD (±½ coding) | none |
| M1 adaptation | adaptation trace of the 5th sound | τ ∈ {0.25 … 32 s} |
| M2 local learner | surprise of the 4→5 transition | half-life ∈ {10, 50, 200, ∞} transitions × reset |
| M3 global learner | surprise of the quintlet's 5th-sound change | half-life ∈ {2, 5, 10, 20, ∞} quintlets × reset |
| M4 / M5 | M2 + M3 / M1 + M3 | both sets |

**Score** = summed out-of-sample Gaussian log-likelihood from *nested* leave-one-block-out CV: each block is
predicted by a model whose regression weights **and** parameters were chosen on the other seven blocks only
(inner LOBO), so flexibility is not rewarded and no penalty term is needed.

In [ ]:
# ---------------------------------------------------------------- the model-comparison machinery (as 06)
H_LOCAL, H_GLOBAL, RESETS = [10, 50, 200, np.inf], [2, 5, 10, 20, np.inf], [False, True]
TAUS = [0.25, 0.5, 1, 2, 4, 8, 16, 32]
COMPONENTS = ["local_80-140ms", "global_300-700ms"]

def regressor_bank(subject):
    '''Every parameterised regressor, one value per presented quintlet (the 5th sound).'''
    qs, tz = load_sequence(subject)
    lo, lr, go, gr = learner_outcomes(qs)
    sid = np.array([c for q in qs["quintlet"] for c in q])
    bank = {}
    for h, r in itertools.product(H_LOCAL, RESETS):
        bank[("loc", h, r)] = leaky_surprise(lo, h, lr if r else None)[3::4]
    for h, r in itertools.product(H_GLOBAL, RESETS):
        bank[("glob", h, r)] = leaky_surprise(go, h, gr if r else None)
    for tau in TAUS:
        bank[("adapt", tau)] = adaptation_trace(tz, sid, tau)[4::5]
    return bank

def model_configs(model, bank):
    loc = [k for k in bank if k[0] == "loc"]; glob = [k for k in bank if k[0] == "glob"]
    ad = [k for k in bank if k[0] == "adapt"]
    return {"M0 categories": [(("cat",), ["LD", "GD", "LDxGD"])],
            "M1 adaptation": [((k,), [k]) for k in ad], "M2 local": [((k,), [k]) for k in loc],
            "M3 global": [((k,), [k]) for k in glob],
            "M4 local+global": [((a, b), [a, b]) for a in loc for b in glob],
            "M5 adapt+global": [((a, b), [a, b]) for a in ad for b in glob]}[model]

def design_matrix(trials, bank, features):
    '''Intercept + standardised session drift + the model's regressors (standardised). Standardising a
    regressor is an affine change absorbed by the intercept/slope, so it does not leak information.'''
    nT = len(trials)
    cols = [np.ones(nT), (trials["seq_idx"] - trials["seq_idx"].mean()) / trials["seq_idx"].std()]
    ld = np.where(trials["local"] == "LD", 0.5, -0.5); gd = np.where(trials["global"] == "GD", 0.5, -0.5)
    for f in features:
        if f == "LD": cols.append(ld)
        elif f == "GD": cols.append(gd)
        elif f == "LDxGD": cols.append(4 * ld * gd)
        else:
            x = bank[f][trials["seq_idx"].to_numpy()]
            cols.append((x - x.mean()) / (x.std() + 1e-12))
    return np.column_stack(cols)

def block_stats(X, y, blocks, ub):
    '''Per-block sufficient statistics (X'X, X'y, y'y, n) so every train/test split is a cheap sum.'''
    return {b: (X[blocks == b].T @ X[blocks == b], X[blocks == b].T @ y[blocks == b],
                float(y[blocks == b] @ y[blocks == b]), int((blocks == b).sum())) for b in ub}

def fit_eval(st, train, test):
    '''OLS on the training blocks; Gaussian log-likelihood of the test blocks with the training noise variance.'''
    XtX = sum(st[b][0] for b in train); Xty = sum(st[b][1] for b in train)
    yty = sum(st[b][2] for b in train); nn = sum(st[b][3] for b in train)
    p = XtX.shape[0]
    beta = np.linalg.solve(XtX + 1e-9 * np.eye(p), Xty)
    s2 = max((yty - 2 * beta @ Xty + beta @ XtX @ beta) / (nn - p), 1e-12)
    ll = sum(-0.5 * st[b][3] * np.log(2 * np.pi * s2) - (st[b][2] - 2 * beta @ st[b][1] + beta @ st[b][0] @ beta) / (2 * s2)
             for b in test)
    return ll, beta, s2

def nested_cv(trials, y, bank, model):
    '''Outer leave-one-block-out prediction; parameters chosen by inner leave-one-block-out within the
    seven training blocks. Returns the summed held-out log-likelihood and the configuration chosen per fold.'''
    blocks = trials["block"].to_numpy(); ub = np.unique(blocks)
    configs = model_configs(model, bank)
    st = [block_stats(design_matrix(trials, bank, feats), y, blocks, ub) for _, feats in configs]
    total, chosen = 0.0, []
    for bo in ub:
        train = [b for b in ub if b != bo]
        best = 0 if len(configs) == 1 else int(np.argmax(
            [sum(fit_eval(s, [b for b in train if b != bi], [bi])[0] for bi in train) for s in st]))
        total += fit_eval(st[best], train, [bo])[0]
        chosen.append(configs[best][0])
    return total, chosen

def load_trials(subject):
    return pd.read_csv(MC_DIR / "trials" / f"{subject}.tsv", sep="\t")

### 2.1 Reproducibility check: refit one participant from scratch

The saved scores (`cv_loglik.tsv`) were produced by `06_model_comparison.py`. Here all six models are refitted
for one participant with the functions above; the two must agree to numerical precision.

In [ ]:
cv_saved = pd.read_csv(MC_DIR / "cv_loglik.tsv", sep="\t")
check_sub = "sub-01"
t0 = time.time()
tr, bank = load_trials(check_sub), regressor_bank(check_sub)
rows = []
for comp in COMPONENTS:
    for m in MODELS:
        ll, _ = nested_cv(tr, tr[comp].to_numpy(), bank, m)
        saved = cv_saved[(cv_saved.subject == check_sub) & (cv_saved.component == comp) & (cv_saved.model == m)].cv_ll.iloc[0]
        rows.append(dict(component=comp, model=m, refit=ll, saved=saved, abs_diff=abs(ll - saved)))
chk = pd.DataFrame(rows)
print(chk.round(4).to_string(index=False)); print(f"({time.time()-t0:.0f} s)")
assert chk.abs_diff.max() < 1e-6, "refit does not reproduce the saved scores"
print("OK: refit reproduces the saved cross-validated scores exactly")

if RECOMPUTE_ALL_CV:
    rows = []
    for s in sorted(p.stem for p in (MC_DIR / "trials").glob("sub-*.tsv")):
        tr, bank = load_trials(s), regressor_bank(s)
        for comp in COMPONENTS:
            for m in MODELS:
                rows.append(dict(component=comp, subject=s, model=m, cv_ll=nested_cv(tr, tr[comp].to_numpy(), bank, m)[0]))
    cv_saved = pd.DataFrame(rows)

### 2.2 Group model comparison (Figure 2D) and head-to-head evidence (Figure 2C)

In [ ]:
win_name = {"local_80-140ms": "early", "global_300-700ms": "late"}
bms = {}
for comp in COMPONENTS:
    Lm = cv_saved[cv_saved.component == comp].pivot(index="subject", columns="model", values="cv_ll")[MODELS]
    b = rfx_bms(Lm.to_numpy()); bms[comp] = b
    best = pd.Series(np.array(MODELS)[Lm.to_numpy().argmax(1)]).value_counts().to_dict()
    print(f"\n{win_name[comp].upper()} window ({comp})")
    print(pd.DataFrame(dict(PXP=b["pxp"], EP=b["ep"], expected_freq=b["expected_freq"]), index=MODELS).round(3).to_string())
    print(f"  BOR = {b['bor']:.3f};  best model per participant: {best}")
    # sensitivity: preregistered six models vs M0-M3 only
    b4 = rfx_bms(Lm[MODELS[:4]].to_numpy())
    print(f"  M0-M3 only: PXP {dict(zip([m.split()[0] for m in MODELS[:4]], b4['pxp'].round(3)))}, BOR = {b4['bor']:.3f}")
    winner = "M2 local" if comp.startswith("local") else "M3 global"
    report(2, f"PXP {winner} ({win_name[comp]})", b["pxp"][MODELS.index(winner)]); report(2, f"BOR {win_name[comp]}", b["bor"])
    report(2, f"PXP {winner} M0-M3 only ({win_name[comp]})", b4["pxp"][MODELS.index(winner)])
    # head-to-head: winner minus each rival, per participant
    other = "M3 global" if winner == "M2 local" else "M2 local"
    for rival in ["M0 categories", "M1 adaptation", other]:
        d = Lm[winner] - Lm[rival]
        print(f"  {winner.split()[0]} - {rival.split()[0]}: mean {d.mean():+.1f} per participant, {int((d > 0).sum())}/{len(d)} favour {winner.split()[0]}")
        report(2, f"{win_name[comp]} {winner.split()[0]}-{rival.split()[0]} mean dLL", d.mean())

### 2.3 Model recovery (Figure 2B)

From `06_model_comparison.py --recovery`: data simulated from each model with each participant's own fitted
parameters, betas and residual noise, then analysed with the full pipeline; 20 simulations per generating model.
Rows = generating model, columns = model with the highest PXP.

In [ ]:
rec = pd.read_csv(MC_DIR / "recovery.tsv", sep="\t")
for comp in COMPONENTS:
    g = rec[rec.component == comp]
    M = pd.crosstab(g.generating, g.winner_pxp, normalize="index").reindex(index=MODELS, columns=MODELS, fill_value=0) * 100
    print(f"\n{win_name[comp]} window: % of simulations won (n = {g.groupby('generating').size().iloc[0]} per model)")
    print(M.round(0).astype(int).rename(columns=lambda c: c.split()[0], index=lambda c: c.split()[0]).to_string())
    for m in ["M2 local", "M3 global"]:
        report(2, f"recovery {m.split()[0]} {win_name[comp]} (%)", M.loc[m, m])

### 2.4 What each learner predicts for the four conditions (Figure 2E)

Within each participant, each learner's surprise is mapped to single-trial window amplitude by OLS (intercept +
slope) and averaged per 2 × 2 cell (written by 07 into `sub-*_cond4.tsv`). The correlation over the four group
means is **descriptive** (four points).

In [ ]:
for comp in ["local", "global"]:
    q = c4[c4.component == comp]
    dat = q[q.source == "data"].pivot(index="subject", columns="condition", values="amp")[CONDS].mean()
    for lname in ["local learner", "global learner"]:
        pr = q[q.source == lname].pivot(index="subject", columns="condition", values="amp")[CONDS].mean()
        r_ = np.corrcoef(pr, dat)[0, 1]
        print(f"{comp:6s} window, {lname:14s}: r = {r_:.2f}   (predicted {pr.round(2).to_dict()})")
        report(2, f"cond-mean r {comp} window {lname}", r_)

### 2.5 Amplitude as a function of model surprise (Figure 2F and unbinned tests)

**(a) Binned (Figure 2F).** Deviants (LD for the early window, GD for the late one) split into within-participant
thirds of the winning learner's surprise; slope of window amplitude over thirds, t-tested across participants.
`raw` = surprise as is; `within` = surprise centred within each 2 × 2 condition (only trial-by-trial learning left).

**(b) Unbinned.** Single-trial OLS of window amplitude on surprise per participant; slopes (µV/bit) t-tested.
The *added-variable* version residualises amplitude and the other learner's surprise on the window's own learner,
i.e. asks what the other learner explains **beyond** it.

**(c) Tie-breaking check.** 07 formed thirds with `rank(method="first")`; when surprise values tie (common with a
block reset — the k-th deviant of every block can have identical surprise), ties are broken by trial order, which
could confound surprise with time in session. The binned slopes are recomputed here from the single-trial data
with ties broken at random (100 permutations) to check this.

In [ ]:
# (a) binned slopes, as written by 07 (sub-*_windows.tsv)
win = pd.concat([pd.read_csv(f, sep="\t") for f in sorted(ERP_DIR.glob("sub-*_windows.tsv"))])
for dev, var, comp in [("LD", "S_loc", "local"), ("GD", "S_glob", "global")]:
    for kind in ["raw", "within"]:
        q = win[(win.deviants == dev) & (win["var"] == var) & (win.kind == kind) & (win.component == comp)]
        piv = q.pivot(index="subject", columns="bin", values="amp")
        sl = piv.apply(lambda r: np.polyfit(np.arange(3), r.to_numpy(), 1)[0], axis=1)
        t = stats.ttest_1samp(sl, 0)
        print(f"{dev} by {var} ({kind:6s}), {comp} window: slope {sl.mean():+.2f} µV/third, t({len(sl)-1}) = {t.statistic:.2f}, "
              f"p = {t.pvalue:.3g}, {int((np.sign(sl)==np.sign(sl.mean())).sum())}/{len(sl)}")
        report(2, f"binned slope t {comp} {kind}", t.statistic)

In [ ]:
# (b) unbinned single-trial slopes, and (c) binned slopes with random tie-breaking
clocks = pd.read_csv(MC_DIR / "clocks.tsv", sep="\t")
subs_mc = sorted(p.stem for p in (MC_DIR / "trials").glob("sub-*.tsv"))
trials = {s: load_trials(s).merge(learner_surprise(s, clocks), on="seq_idx", validate="one_to_one") for s in subs_mc}
WIN_COL = {"local": "local_80-140ms", "global": "global_300-700ms"}

def resid(v, z):
    Z = np.column_stack([np.ones(len(z)), z])
    return v - Z @ np.linalg.lstsq(Z, v, rcond=None)[0]

def slopes(comp, var, mode, control=None):
    out = []
    for s, d in trials.items():
        y, x = d[WIN_COL[comp]].to_numpy(float), d[var].to_numpy(float)
        if mode == "within":     # centre both within each 2x2 condition
            x = x - d.groupby("condition")[var].transform("mean").to_numpy()
            y = y - d.groupby("condition")[WIN_COL[comp]].transform("mean").to_numpy()
        if mode == "partial":    # added-variable: residualise on the window's own learner
            z = d[control].to_numpy(float); x, y = resid(x, z), resid(y, z)
        out.append(np.polyfit(x, y, 1)[0])
    out = np.array(out); t = stats.ttest_1samp(out, 0)
    return out.mean(), t.statistic, t.pvalue, int((np.sign(out) == np.sign(out.mean())).sum())

for comp, own, other in [("local", "S_loc", "S_glob"), ("global", "S_glob", "S_loc")]:
    for label, args in [("own learner", (own, "raw")), ("own learner, within condition", (own, "within")),
                        ("other learner beyond own", (other, "partial", own))]:
        m, t, p, k = slopes(comp, *args)
        print(f"{comp:6s} window, {label:32s}: {m:+.2f} µV/bit, t(14) = {t:.2f}, p = {p:.3g}, {k}/15")
        report(2, f"single-trial slope t {comp} {label}", t)

rng = np.random.default_rng(SEED)
def tertile_slopes(comp, var, dev_col, dev, within, tie="first"):
    out = []
    for s, d in trials.items():
        q = d[d[dev_col] == dev].reset_index(drop=True)
        x = q[var].to_numpy(float); y = q[WIN_COL[comp]].to_numpy(float)
        if within:
            x = x - q.groupby("condition")[var].transform("mean").to_numpy()
        key = pd.Series(x + (rng.uniform(0, 1e-9, len(x)) if tie == "random" else 0)).rank(method="first")
        b = pd.qcut(key, 3, labels=False).to_numpy()
        out.append(np.polyfit(np.arange(3), [y[b == k].mean() for k in range(3)], 1)[0])
    return stats.ttest_1samp(out, 0).statistic

for comp, var, dev_col, dev in [("local", "S_loc", "local", "LD"), ("global", "S_glob", "global", "GD")]:
    for within in [False, True]:
        frac_tied = np.mean([1 - d[d[dev_col] == dev][var].nunique() / (d[dev_col] == dev).sum() for d in trials.values()])
        t_first = tertile_slopes(comp, var, dev_col, dev, within, "first")
        t_rand = np.array([tertile_slopes(comp, var, dev_col, dev, within, "random") for _ in range(100)])
        print(f"{comp:6s} {'within' if within else 'raw   '}: tied values {100*frac_tied:4.1f}%  t(order tie-break) = {t_first:+.2f}  "
              f"t(random tie-break) = {t_rand.mean():+.2f} [{np.percentile(t_rand,2.5):+.2f}, {np.percentile(t_rand,97.5):+.2f}]")

### 2.6 Timescales of the two learners (preregistered "clocks" test) and block reset

**Timescales.** Each learner's selected half-life (from the full-data LOBO fit) is converted to seconds
(local: transitions ÷ 4 × quintlet period; global: quintlets × period; ∞ = no forgetting, ranked highest) and
compared between levels with a Wilcoxon signed-rank test. Preregistered criterion: p < .05.

**Reset** (from `05_reset_validation.py`, which needs the epochs): *Test 1* — a learner that carries memory across
blocks predicts a smaller P3b (300–600 ms) to early-block deviants whose structure was the previous block's
standard; tested with a paired t-test and JZS Bayes factor. *Test 2* — the global learner with memory scaled by
ρ at each block start (ρ = 0 full reset, 1 none), leave-one-block-out CV over ρ. Both are recomputed here from the
saved per-participant tables.

In [ ]:
c_loc = clocks[(clocks.component == "local_80-140ms") & (clocks.model == "M2 local")].set_index("subject")
c_glo = clocks[(clocks.component == "global_300-700ms") & (clocks.model == "M3 global")].set_index("subject")
a = c_loc["half_life_s"].replace(np.inf, 1e6); b = c_glo["half_life_s"].loc[a.index].replace(np.inf, 1e6)
w_ = stats.wilcoxon(a, b)
print(f"half-life (s): local median {c_loc.half_life_s.median():.2f}, global median {c_glo.half_life_s.median():.2f}; "
      f"Wilcoxon W = {w_.statistic:.1f}, p = {w_.pvalue:.3f}; local longer in {(a > b).sum()}, equal in {(a == b).sum()}")
print(f"reset selected: local learner {int(c_loc.reset.sum())}/15, global learner {int(c_glo.reset.sum())}/15")
report(2, "clocks Wilcoxon p", w_.pvalue)

t1 = pd.read_csv(RV_DIR / "test1_history.tsv", sep="\t")
w1 = t1.pivot(index="subject", columns="history", values="p3").dropna()
d1 = w1["was_prev_standard"] - w1["other"]; tt1 = stats.ttest_rel(w1["was_prev_standard"], w1["other"])
bf10 = jzs_bf10(d1.to_numpy())
print(f"\nReset test 1: P3b difference {d1.mean():+.2f} µV, t({len(d1)-1}) = {tt1.statistic:.2f}, p = {tt1.pvalue:.2f}, BF01 = {1/bf10:.2f}")
report(2, "reset test1 BF01", 1 / bf10)

t2 = pd.read_csv(RV_DIR / "test2_rho_cv.tsv", sep="\t")
prof = t2.pivot(index="subject", columns="rho", values="cv_loglik")
boot = [prof.iloc[rng.integers(0, len(prof), len(prof))].sum().idxmax() for _ in range(5000)]
best = prof.idxmax(axis=1)
r2 = t2.pivot(index="subject", columns="rho", values="cv_r2"); tt2 = stats.ttest_rel(r2[0.0], r2[1.0])
print(f"Reset test 2: group-best rho = {prof.sum().idxmax()}, bootstrap 95% CI [{np.percentile(boot,2.5)}, {np.percentile(boot,97.5)}]; "
      f"per-participant best rho <= 0.1 in {(best <= 0.1).sum()}/15 (rho = 1 in {(best == 1).sum()}/15); "
      f"CV R² reset > no reset in {(r2[0.0] > r2[1.0]).sum()}/15, t = {tt2.statistic:.2f}, p = {tt2.pvalue:.3f}")
report(2, "reset group-best rho", prof.sum().idxmax())

### Figure 2A–B — model schematic and regressors

`scripts/15_figure2_schematic.py`. From the script's header:

```text
15_figure2_schematic.py — Figure 2 schematic at manuscript size (173.695 x 65 mm).

  A  Concept matrix: the three candidate mechanisms (adaptation, local transition learning,
     global rule learning) plus fixed category labels as pictograms; each model (row) is the
     set of components it contains (filled dots)
  B  The regressors on real trials around a rule switch (one subject), sharing one x-axis:
     M0 category labels, M1 adaptation trace, M2 local surprise, M3 global surprise.
     (M4/M5 are sums of these, so they are not drawn separately.)

Illustrative parameters = group-typical values from 06_model_comparison.py
(adaptation tau = 8 s; local learner: no forgetting within block, reset; global learner:
half-life 10 quintlets, reset). The task itself is in Figure 1, so it is not repeated here.

Needs only sourcedata (no EEG). Editable text: TrueType in PDF, <text> in SVG (font named Arial).
Output: figures/Figure2_schematic.pdf / .svg / .png

Usage:  python scripts/15_figure2_schematic.py [sub-01]
```

In [ ]:
# ---- 15_figure2_schematic.py (inline; rebuild the notebook after editing the script)
def make_figure2_schematic():
    from pathlib import Path
    import sys

    import numpy as np
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch, Rectangle
    import seaborn as sns
    from scipy.io import loadmat

    # ----------------------------------------------------------------------------- CONFIG
    ROOT = PROJECT_ROOT
    BIDS_ROOT = ROOT / "data" / "bids"
    FIG_DIR = ROOT / "figures"
    SOUNDS = {1: "A", 2: "B"}
    SUBJECT = "sub-01"
    SIZE_MM = (173.695, 65.0)
    SWITCH_BLOCK = 2                 # show the end of block 1 and the start of block 2
    N_BEFORE, N_AFTER = 20, 40       # quintlets either side of the switch
    TAU, H_LOC, RESET_LOC, H_GLOB, RESET_GLOB = 8.0, np.inf, True, 10, True
    MCOL = {"categories": "0.35", "adaptation": "#55A868", "local": "#C44E52", "global": "#8172B3",
            "local+global": "#937860", "adapt+global": "#64B5CD"}
    TEXT_RC = {"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
               "font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "Helvetica", "DejaVu Sans"],
               "font.size": 7, "axes.labelsize": 6.5, "xtick.labelsize": 6, "ytick.labelsize": 6,
               "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
               "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5,
               "axes.labelpad": 2}
    TONE = {"A": "#4C72B0", "B": "#DD8452"}
    CARDS = [  # name, colour key, pictogram(s), short gist
        ("M0  Categories", "categories", ["categories"], "fixed labels, no learning"),
        ("M1  Adaptation", "adaptation", ["adaptation"], "sensory fatigue"),
        ("M2  Local", "local", ["local"], "sound-to-sound prediction"),
        ("M3  Global", "global", ["global"], "rule prediction"),
        ("M4  Local + global", "local+global", ["local", "global"], "both predictions"),
        ("M5  Adapt. + global", "adapt+global", ["adaptation", "global"], "fatigue + rule"),
    ]
    # ------------------------------------------------------------------------------------


    def load(subject):
        src = BIDS_ROOT / "sourcedata" / subject / "ses-LGeffect"
        seq = loadmat(next(src.glob("GlobLoc_stimulation_SEQUENCE*.mat")), squeeze_me=True)["SEQUENCE"]
        tz = loadmat(next(src.glob("GlobLoc_critical_events*.mat")), squeeze_me=True)["tz"]
        tz = tz[tz > 0]
        rows = []
        for b, blk in enumerate(seq, start=1):
            blk = np.atleast_2d(blk).astype(int)
            std = "".join(SOUNDS[x] for x in blk[0])
            for i, q in enumerate(blk):
                rows.append(dict(block=b, trial_in_block=i, quintlet="".join(SOUNDS[x] for x in q), std=std))
        qs = pd.DataFrame(rows)
        qs["local"] = np.where(qs["quintlet"].str[3] == qs["quintlet"].str[4], "LS", "LD")
        qs["global"] = np.where(qs["quintlet"] == qs["std"], "GS", "GD")
        return qs, tz


    def leaky(outcomes, h, resets):
        """Leaky Beta-Bernoulli learner; returns surprise (bits) and p(change) before each outcome."""
        lam = 1.0 if np.isinf(h) else 2.0 ** (-1.0 / h)
        n1 = n0 = 0.0
        S, MU = np.empty(len(outcomes)), np.empty(len(outcomes))
        for t, y in enumerate(outcomes):
            if resets is not None and resets[t]:
                n1 = n0 = 0.0
            mu = (n1 + 1) / (n1 + n0 + 2)
            MU[t], S[t] = mu, -np.log2(mu if y else 1 - mu)
            n1, n0 = lam * n1 + y, lam * n0 + (1 - y)
        return S, MU


    def adaptation(times, ids, tau):
        """Stimulus-specific adaptation trace of each sound just before it is played."""
        a = {"A": 0.0, "B": 0.0}
        last, out = times[0], np.empty(len(times))
        for i, (t, k) in enumerate(zip(times, ids)):
            dec = np.exp(-(t - last) / tau)
            a["A"] *= dec
            a["B"] *= dec
            out[i] = a[k]
            a[k] += 1
            last = t
        return out


    def regressors(qs, tz):
        first, fifth = qs["quintlet"].str[0].to_numpy(), qs["quintlet"].str[4].to_numpy()
        start = (qs["trial_in_block"] == 0).to_numpy()
        lo = np.zeros((len(qs), 4), int)          # 4 transitions per quintlet; only 4->5 can change
        lo[:, 3] = fifth != first
        lo = lo.ravel()
        lr = np.zeros(len(lo), bool)
        lr[::4] = start
        S_loc, _ = leaky(lo, H_LOC, lr if RESET_LOC else None)
        S_glob, _ = leaky((fifth != first).astype(int), H_GLOB, start if RESET_GLOB else None)
        ids = np.array([c for q in qs["quintlet"] for c in q])
        qs = qs.copy()
        qs["S_loc"], qs["S_glob"] = S_loc[3::4], S_glob
        qs["adapt"] = adaptation(tz, ids, TAU)[4::5]
        return qs


    def _quint(ax, x, y, q, w, gap, h):
        for c in q:
            ax.add_patch(Rectangle((x, y), w, h if c == "A" else h * 1.4, fc=TONE[c], ec="none"))
            x += w + gap
        return x


    def pictogram(ax, kind, x0, y0, w, h):
        """Conceptual icon for each model component, drawn inside the box (x0, y0, w, h) in inches."""
        if kind == "categories":                          # 2 x 2 table of fixed levels
            s = min(w, h) * 0.38
            cx, cy = x0 + w / 2 + 0.04, y0 + h / 2 - 0.03
            for i, lv in enumerate([0.95, 0.75, 0.6, 0.35]):
                r, c = divmod(i, 2)
                ax.add_patch(Rectangle((cx - s + c * s, cy - r * s), s * 0.94, s * 0.94, fc=str(lv), ec="none"))
            if w < 0.8:
                return
            ax.text(cx - s / 2, cy + s + 0.03, "LS", ha="center", fontsize=5, color="0.4")
            ax.text(cx + s / 2, cy + s + 0.03, "LD", ha="center", fontsize=5, color="0.4")
            ax.text(cx - s - 0.03, cy + s / 2, "GS", ha="right", va="center", fontsize=5, color="0.4")
            ax.text(cx - s - 0.03, cy - s / 2, "GD", ha="right", va="center", fontsize=5, color="0.4")
        elif kind == "adaptation":                        # trace jumps at each sound, decays between
            t = np.linspace(0, 1, 300)
            ev = [0.08, 0.16, 0.24, 0.32, 0.62, 0.70]
            tr = np.zeros_like(t)
            for e in ev:
                tr += (t >= e) * np.exp(-(t - e) / 0.18)
            tr /= tr.max()
            ax.plot(x0 + 0.04 + t * (w - 0.08), y0 + 0.08 + tr * (h - 0.2), color=MCOL["adaptation"], lw=0.9)
            for e in ev:
                ax.plot([x0 + 0.04 + e * (w - 0.08)] * 2, [y0 + 0.02, y0 + 0.07], color="0.3", lw=0.7)
            ax.plot([x0 + 0.04, x0 + w - 0.04], [y0 + 0.045, y0 + 0.045], color="0.6", lw=0.4)
        elif kind == "local":                             # five sounds; arcs between neighbours
            tw, gap = w * 0.1, w * 0.07
            xs0 = x0 + (w - (5 * tw + 4 * gap)) / 2
            yb = y0 + h * 0.18
            _quint(ax, xs0, yb, "AAAAB", tw, gap, h * 0.28)
            for k in range(4):
                xa, xb = xs0 + k * (tw + gap) + tw / 2, xs0 + (k + 1) * (tw + gap) + tw / 2
                last = k == 3
                ax.add_patch(matplotlib.patches.Arc(((xa + xb) / 2, yb + h * 0.45), xb - xa, h * 0.28, theta1=0,
                                                    theta2=180, color=MCOL["local"] if last else "0.55",
                                                    lw=1.1 if last else 0.6))
            ax.text(xs0 + 3.5 * (tw + gap) + tw / 2 - gap / 2, yb + h * 0.66, "?", ha="center", fontsize=7,
                    color=MCOL["local"], weight="bold")
        elif kind == "global":                            # whole quintlets; the rule is learned across them
            tw, gap, qg = w * 0.028, w * 0.012, w * 0.06
            qw = 5 * tw + 4 * gap
            xs0 = x0 + (w - (3 * qw + 2 * qg)) / 2
            yb = y0 + h * 0.18
            for k, q in enumerate(["AAAAA", "AAAAA", "AAAAB"]):
                _quint(ax, xs0 + k * (qw + qg), yb, q, tw, gap, h * 0.28)
            xe = xs0 + 2 * (qw + qg)
            ax.plot([xs0, xs0, xe - qg * 0.5, xe - qg * 0.5], [yb + h * 0.5, yb + h * 0.56, yb + h * 0.56, yb + h * 0.5],
                    color=MCOL["global"], lw=0.8)
            ax.text((xs0 + xe) / 2 - qg / 2, yb + h * 0.6, "rule", ha="center", fontsize=5.5, color=MCOL["global"])
            ax.text(xe + qw / 2, yb + h * 0.48, "?", ha="center", fontsize=7, color=MCOL["global"], weight="bold")


    COMPONENTS = [("categories", "categories"), ("adaptation", "adaptation"), ("local", "local"),
                  ("global", "global")]           # (pictogram, colour key) for the matrix columns
    MODEL_SETS = [("M0", "categories", {"categories"}), ("M1", "adaptation", {"adaptation"}),
                  ("M2", "local", {"local"}), ("M3", "global", {"global"})]
    # M4 (local + global) and M5 (adaptation + global) were also fitted (preregistered) but are not
    # separable from their components in model recovery; they are named in the legend, not drawn here.


    def draw_matrix(ax, W, H):
        """Panel A in inch coordinates: component pictograms as columns, models as rows, dots = included."""
        lab_w, icon_h, top = 0.3, 0.72, H - 0.12
        cw = (W - lab_w) / len(COMPONENTS)
        rh = (top - icon_h - 0.1) / len(MODEL_SETS)
        for j, (pic, ck) in enumerate(COMPONENTS):
            x0 = lab_w + j * cw
            ax.add_patch(FancyBboxPatch((x0 + 0.03, top - icon_h), cw - 0.06, icon_h,
                                        boxstyle="round,pad=0,rounding_size=0.04", fc="0.97", ec="none"))
            pictogram(ax, pic, x0 + 0.05, top - icon_h + 0.04, cw - 0.1, icon_h - 0.08)
        for i, (name, mk, comps) in enumerate(MODEL_SETS):
            yc = top - icon_h - 0.1 - (i + 0.5) * rh
            if i % 2 == 0:
                ax.add_patch(Rectangle((0, yc - rh / 2), W, rh, fc="0.965", ec="none", zorder=0))
            ax.text(lab_w - 0.06, yc, name, ha="right", va="center", fontsize=7, weight="bold", color=MCOL[mk])
            on = [j for j, (pic, _) in enumerate(COMPONENTS) if pic in comps]
            if len(on) == 2:
                ax.plot([lab_w + (on[0] + 0.5) * cw, lab_w + (on[1] + 0.5) * cw], [yc, yc], color=MCOL[mk], lw=1.2,
                        zorder=1)
            for j, (pic, ck) in enumerate(COMPONENTS):
                xc = lab_w + (j + 0.5) * cw
                if pic in comps:
                    ax.add_patch(matplotlib.patches.Circle((xc, yc), 0.055, fc=MCOL[mk], ec="none", zorder=2))
                else:
                    ax.add_patch(matplotlib.patches.Circle((xc, yc), 0.028, fc="0.85", ec="none", zorder=2))


    def main(subject):
        qs, tz = load(subject)
        qs = regressors(qs, tz)
        sw = qs.index[(qs["block"] == SWITCH_BLOCK) & (qs["trial_in_block"] == 0)][0]
        sns.set_theme(style="ticks", rc=TEXT_RC)
        W, H = SIZE_MM[0] / 25.4, SIZE_MM[1] / 25.4
        fig = plt.figure(figsize=(W, H))

        # ---------------- A: model cards (inch coordinates)
        wa = 2.55
        axA = fig.add_axes([0.0, 0.0, wa / W, 1.0])
        axA.set_axis_off()
        axA.set_xlim(-0.02, wa)
        axA.set_ylim(0, H)
        draw_matrix(axA, wa - 0.06, H)
        fig.text(0.0, 0.995, "A", fontsize=9, weight="bold", va="top")

        # ---------------- B: regressors on real trials around a rule switch
        seg = qs.loc[sw - N_BEFORE: sw + N_AFTER - 1]
        xs = np.arange(len(seg)) - N_BEFORE
        gd = seg["global"].eq("GD").to_numpy()
        ld = seg["local"].eq("LD").to_numpy()
        rows = [("categories", "M0", None), ("adaptation", "M1", "adapt"),
                ("local", "M2", "S_loc"), ("global", "M3", "S_glob")]
        x_left, x_right, y_bot, y_top = (wa + 0.42) / W, 0.995, 0.17, 0.90
        h = (y_top - y_bot) / len(rows)
        axes = []
        for r, (key, lab, col) in enumerate(rows):
            a = fig.add_axes([x_left, y_top - (r + 1) * h + 0.012, x_right - x_left, h - 0.024],
                             sharex=axes[0] if axes else None)
            axes.append(a)
            if key == "categories":
                a.scatter(xs[ld], np.full(ld.sum(), 1), marker="|", s=18, lw=0.9, color=MCOL["local"])
                a.scatter(xs[gd], np.full(gd.sum(), 0), marker="|", s=18, lw=0.9, color=MCOL["global"])
                a.set_yticks([0, 1], ["GD", "LD"])
                a.set_ylim(-0.7, 1.7)
            else:
                y = seg[col].to_numpy()
                a.plot(xs, y, color=MCOL[key], lw=0.8, marker="o", ms=1.6, mew=0)
                a.scatter(xs[gd], y[gd], s=9, facecolor="none", edgecolor="k", lw=0.5, zorder=3)
                a.set_yticks([])
                a.spines["left"].set_visible(False)
            a.axvline(-0.5, color="k", lw=0.6, ls="--")
            a.set_ylabel(lab, rotation=0, ha="right", va="center", fontsize=6.5, color=MCOL[key], weight="bold")
            a.set_xlim(xs[0] - 1, xs[-1] + 1)
            if r < len(rows) - 1:
                plt.setp(a.get_xticklabels(), visible=False)
            sns.despine(ax=a)
        axes[-1].set_xticks([-20, -10, 0, 10, 20, 30])
        axes[-1].set_xlabel("Trials from rule switch")
        axes[0].text(-1.0, 1.9, f"rule: {qs.loc[sw - 1, 'std']}", ha="right", va="bottom", fontsize=5.8,
                     transform=axes[0].transData)
        axes[0].text(0.5, 1.9, f"rule: {qs.loc[sw, 'std']}", ha="left", va="bottom", fontsize=5.8)
        fig.text(wa / W + 0.005, 0.995, "B", fontsize=9, weight="bold", va="top")

        FIG_DIR.mkdir(exist_ok=True)
        fig.savefig(FIG_DIR / "Figure2_schematic.pdf")
        fig.savefig(FIG_DIR / "Figure2_schematic.svg")
        svg = FIG_DIR / "Figure2_schematic.svg"          # SVG names fonts: Liberation Sans -> Arial (same metrics)
        svg.write_text(svg.read_text().replace("Liberation Sans", "Arial"))
        fig.savefig(FIG_DIR / "Figure2_schematic.png", dpi=400)
        print(f"Figure -> {FIG_DIR / 'Figure2_schematic'}.pdf/.svg/.png "
              f"({fig.get_size_inches()[0] * 25.4:.1f} x {fig.get_size_inches()[1] * 25.4:.1f} mm)")
        print("block standards:", qs.loc[sw - 1, "std"], "->", qs.loc[sw, "std"])

    return main(SUBJECT)


run_figure('Figure 2A–B', make_figure2_schematic)

plt.close('all')
for f in ['Figure2_schematic.png']:
    p = PROJECT_ROOT / 'figures' / f
    display(Image(filename=str(p), width=900) if p.exists() else f'missing: {p}')

### Figure 2C–F — model comparison results

`scripts/16_figure2_results.py`. From the script's header:

```text
16_figure2_results.py — Figure 2 results panels at manuscript size (173.695 x 80 mm).

Rows = response windows (early frontocentral 80–140 ms; late parietal 300–700 ms).
Columns:
  C  head-to-head evidence: held-out log-likelihood of the window's winning learner minus each rival
     (M0 categories, M1 adaptation, the other learner), one dot per participant; n/15 = participants > 0
  D  group verdict: protected exceedance probability (random-effects BMS) over the preregistered six
     models; M4/M5 faded (not separable from their components in model recovery)
  E  what the winner explains: window amplitude per 2x2 condition (bars = data, mean ± SE) with the
     local and global learners' predictions (each learner's surprise mapped to single-trial amplitude
     by within-subject OLS; from 07_header1_erps.py), and the agreement r across the 4 conditions
  F  example ERP: that window's deviants split into per-participant tertiles of the winning learner's
     surprise (LD by local surprise at the frontocentral ROI; GD by global surprise at the parietal ROI),
     with the standard (dashed); slope = per-participant linear trend of window amplitude across tertiles

Inputs: derivatives/model_comparison/{cv_loglik.tsv, summary.json}
        derivatives/header1_erps/sub-*_cond4.tsv, sub-*_windows.tsv, sub-*.npz (from 07_header1_erps.py)
Output: figures/Figure2_results.pdf / .svg / .png (editable text; SVG font named Arial)

Usage:  python scripts/16_figure2_results.py
```

In [ ]:
# ---- 16_figure2_results.py (inline; rebuild the notebook after editing the script)
def make_figure2_results():
    from pathlib import Path
    import json

    import numpy as np
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy import stats

    # ----------------------------------------------------------------------------- CONFIG
    ROOT = PROJECT_ROOT
    MC_DIR = ROOT / "derivatives" / "model_comparison"
    ERP_DIR = ROOT / "derivatives" / "header1_erps"
    FIG_DIR = ROOT / "figures"
    SIZE_MM = (173.695, 80.0)
    MODELS = ["M0 categories", "M1 adaptation", "M2 local", "M3 global", "M4 local+global", "M5 adapt+global"]
    SHORT = {m: m.split()[0] for m in MODELS}
    MCOL = {"M0 categories": "0.35", "M1 adaptation": "#55A868", "M2 local": "#C44E52", "M3 global": "#8172B3",
            "M4 local+global": "#937860", "M5 adapt+global": "#64B5CD"}
    ROWS = [  # model-comparison component, ERP key, row label, winning model, binned-ERP spec (deviants, surprise)
        ("local_80-140ms", "local", "Early\n80–140 ms", "M2 local", ("LD", "S_loc", "LS")),
        ("global_300-700ms", "global", "Late\n300–700 ms", "M3 global", ("GD", "S_glob", "GS"))]
    WIN = {"local": (0.08, 0.14), "global": (0.30, 0.70)}
    BIN_PAL = {"local": ["#F2B8B9", "#D9666A", "#8E1F24"], "global": ["#CFCBE8", "#9890CB", "#4E4596"]}
    BIN_KIND = "raw"          # tertiles of the learner's surprise on that window's deviant trials
    CONDS = ["LSGS", "LDGS", "LSGD", "LDGD"]
    COND_LAB = ["LS\nGS", "LD\nGS", "LS\nGD", "LD\nGD"]
    LEARNERS = [("local learner", "M2 local", "D"), ("global learner", "M3 global", "o")]
    TEXT_RC = {"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
               "font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "Helvetica", "DejaVu Sans"],
               "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 6.5, "xtick.labelsize": 6, "ytick.labelsize": 6,
               "legend.fontsize": 6, "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
               "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5,
               "axes.labelpad": 2}
    SEED = 1
    # ------------------------------------------------------------------------------------


    def main():
        ll = pd.read_csv(MC_DIR / "cv_loglik.tsv", sep="\t")
        summ = json.loads((MC_DIR / "summary.json").read_text())
        c4 = pd.concat([pd.read_csv(f, sep="\t") for f in sorted(ERP_DIR.glob("sub-*_cond4.tsv"))])
        n = ll["subject"].nunique()
        rng = np.random.default_rng(SEED)

        sns.set_theme(style="ticks", rc=TEXT_RC)
        mm = 1 / 25.4
        fig = plt.figure(figsize=(SIZE_MM[0] * mm, SIZE_MM[1] * mm))
        gs = fig.add_gridspec(2, 4, width_ratios=[0.62, 0.62, 0.95, 1.05], left=0.105, right=0.99, bottom=0.1, top=0.9,
                              wspace=0.5, hspace=0.42)
        stats_out = {}
        for r, (comp, ek, rowlab, winner, (dev, svar, std)) in enumerate(ROWS):
            # ---------------- C: head-to-head — winner minus each alternative, per participant
            ax = fig.add_subplot(gs[r, 0])
            d = ll[ll["component"] == comp].pivot(index="subject", columns="model", values="cv_ll")
            rivals = ["M0 categories", "M1 adaptation", "M3 global" if winner == "M2 local" else "M2 local"]
            stats_out[comp] = {}
            for j, m in enumerate(rivals):
                y = (d[winner] - d[m]).to_numpy()
                ax.scatter(j + rng.uniform(-0.13, 0.13, n), y, s=6, lw=0, color=MCOL[winner], alpha=0.8, zorder=3)
                ax.errorbar(j + 0.32, y.mean(), yerr=y.std(ddof=1) / np.sqrt(n), fmt="o", ms=2.8, lw=0.8, color="k",
                            zorder=4)
                k = int((y > 0).sum())
                ax.text(j, 1.0, f"{k}/{n}", transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=5.5,
                        color="0.3")
                stats_out[comp][f"{SHORT[winner]}_minus_{SHORT[m]}"] = dict(mean=float(y.mean()), n_positive=k,
                                                                             total=float(y.sum()))
            ax.axhline(0, color="k", lw=0.5)
            ax.set_xticks(range(len(rivals)), [f"vs {SHORT[m]}" for m in rivals])
            for tl, m in zip(ax.get_xticklabels(), rivals):
                tl.set_color(MCOL[m])
            ax.set_xlim(-0.5, len(rivals) - 0.35)
            ax.set_ylabel(f"Δ log-lik. ({SHORT[winner]} − rival)", color=MCOL[winner])
            sns.despine(ax=ax)
            ax.text(-0.62, 0.5, rowlab, transform=ax.transAxes, rotation=90, ha="center", va="center", fontsize=7,
                    weight="bold")
            if r == 0:
                ax.set_title("Winner vs rivals", pad=9)

            # ---------------- D: PXP
            ax = fig.add_subplot(gs[r, 1])
            pxp = [summ[comp]["pxp"][m] for m in MODELS]
            faint = ("M4 local+global", "M5 adapt+global")     # not recoverable at this n: shown, de-emphasised
            ax.bar(range(6), pxp, color=[MCOL[m] for m in MODELS], width=0.7,
                   alpha=None)
            for patch, m in zip(ax.patches, MODELS):
                patch.set_alpha(0.35 if m in faint else 1.0)
            ax.set_xticks(range(6), [SHORT[m] for m in MODELS])
            for tl, m in zip(ax.get_xticklabels(), MODELS):
                tl.set_color(MCOL[m])
                tl.set_alpha(0.45 if m in faint else 1.0)
            win = int(np.argmax(pxp))
            ax.text(win, pxp[win] + 0.03, f"{pxp[win]:.2f}", ha="center", va="bottom", fontsize=6)
            ax.set_ylim(0, 1.12)
            ax.set_yticks([0, 0.5, 1])
            ax.set_ylabel("PXP")
            sns.despine(ax=ax)
            if r == 0:
                ax.set_title("Group verdict", pad=9)
            stats_out[comp].update(pxp=dict(zip(MODELS, pxp)), bor=summ[comp]["bor"])

            # ---------------- E: data vs learner predictions per condition
            ax = fig.add_subplot(gs[r, 2])
            q = c4[c4["component"] == ek]
            dat = q[q["source"] == "data"].pivot(index="subject", columns="condition", values="amp")[CONDS]
            xs = np.arange(4)
            ax.bar(xs, dat.mean(), yerr=dat.std(ddof=1) / np.sqrt(len(dat)), color="0.82", width=0.7,
                   error_kw=dict(lw=0.7, capsize=0), label="data")
            for k, (lname, m, mk) in enumerate(LEARNERS):
                pr = q[q["source"] == lname].pivot(index="subject", columns="condition", values="amp")[CONDS]
                off = -0.13 if k == 0 else 0.13
                rv = float(np.corrcoef(pr.mean(), dat.mean())[0, 1])
                ax.plot(xs + off, pr.mean(), marker=mk, ls="none", ms=3.6, mfc="white", mec=MCOL[m], mew=1.0,
                        label=f"{SHORT[m]}  r = {rv:.2f}")
                stats_out[comp][f"r_{lname}"] = rv
            ax.axhline(0, color="k", lw=0.5)
            ax.set_xticks(xs, COND_LAB)
            ax.set_ylabel("Amplitude (µV)")
            sns.despine(ax=ax)
            ax.legend(frameon=False, loc="lower left" if r == 0 else "upper left", handlelength=1.0,
                      borderaxespad=0.1, handletextpad=0.3, labelspacing=0.3)
            if r == 0:
                ax.set_title("Data vs predictions", pad=9)

            # ---------------- F: example ERP — deviants split by the winning learner's surprise
            ax = fig.add_subplot(gs[r, 3])
            files = sorted(ERP_DIR.glob("sub-*.npz"))
            t = np.load(files[0])["times"] * 1e3
            ref = np.stack([np.load(f)[f"cond__{std}"] for f in files])
            ax.axvspan(WIN[ek][0] * 1e3, WIN[ek][1] * 1e3, color="0.93", lw=0, zorder=0)
            ax.plot(t, ref.mean(0), color="0.55", lw=0.8, ls="--", label=f"{std} (standard)")
            for k, lab in enumerate(["low", "mid", "high"]):
                X = np.stack([np.load(f)[f"bin__{dev}__{svar}__{BIN_KIND}__{ek}__{k}"] for f in files])
                m_, se = X.mean(0), X.std(0, ddof=1) / np.sqrt(len(X))
                ax.plot(t, m_, color=BIN_PAL[ek][k], lw=1.0, label=lab)
                ax.fill_between(t, m_ - se, m_ + se, color=BIN_PAL[ek][k], alpha=0.18, lw=0)
            ax.axvline(0, color="k", lw=0.5)
            ax.axhline(0, color="k", lw=0.4)
            ax.set_xlim(-100, 700)
            ax.set_xticks([0, 200, 400, 600])
            ax.set_xlabel("Time (ms)")
            ax.set_ylabel("Amplitude (µV)")
            sns.despine(ax=ax)
            ax.legend(frameon=False, loc="lower right", handlelength=1.0, ncol=2,
                      borderaxespad=0.1, handletextpad=0.3, labelspacing=0.2, columnspacing=0.6, fontsize=5.5,
                      title=f"{dev} by {SHORT[winner]} surprise", title_fontsize=5.5)
            # per-participant slope of window amplitude across surprise tertiles
            w = pd.concat([pd.read_csv(f, sep="\t") for f in sorted(ERP_DIR.glob("sub-*_windows.tsv"))])
            for kind in ("raw", "within"):
                q = w[(w["deviants"] == dev) & (w["var"] == svar) & (w["kind"] == kind) & (w["component"] == ek)]
                piv = q.pivot(index="subject", columns="bin", values="amp")
                sl = piv.apply(lambda row: np.polyfit(np.arange(3), row.to_numpy(), 1)[0], axis=1)
                tt = stats.ttest_1samp(sl, 0)
                stats_out[comp][f"surprise_slope_{kind}"] = dict(mean_uV_per_tertile=float(sl.mean()), t=float(tt.statistic),
                                                                 p=float(tt.pvalue), n_same_sign=int((np.sign(sl) == np.sign(sl.mean())).sum()))
            sr = stats_out[comp][f"surprise_slope_{BIN_KIND}"]
            ax.text(0.03, 1.0, f"slope t({n - 1}) = {sr['t']:.1f}".replace("-", "−"),
                    transform=ax.transAxes, ha="left", va="top", fontsize=5.5)
            if r == 0:
                ax.set_title("ERP by model surprise", pad=9)

        for lab, ax in zip("CDEF", fig.axes[:4]):
            fig.text(ax.get_position().x0 - 0.05, 0.985, lab, fontsize=9, weight="bold", va="top")

        FIG_DIR.mkdir(exist_ok=True)
        fig.savefig(FIG_DIR / "Figure2_results.pdf")
        fig.savefig(FIG_DIR / "Figure2_results.svg")
        svg = FIG_DIR / "Figure2_results.svg"            # SVG names fonts: Liberation Sans -> Arial (same metrics)
        svg.write_text(svg.read_text().replace("Liberation Sans", "Arial"))
        fig.savefig(FIG_DIR / "Figure2_results.png", dpi=400)
        (MC_DIR / "figure2_results_stats.json").write_text(json.dumps(stats_out, indent=2))
        print(json.dumps(stats_out, indent=2))
        print(f"Figure -> {FIG_DIR / 'Figure2_results'}.pdf/.svg/.png "
              f"({fig.get_size_inches()[0] * 25.4:.1f} x {fig.get_size_inches()[1] * 25.4:.1f} mm)")

    return main()


run_figure('Figure 2C–F', make_figure2_results)

plt.close('all')
for f in ['Figure2_results.png']:
    p = PROJECT_ROOT / 'figures' / f
    display(Image(filename=str(p), width=900) if p.exists() else f'missing: {p}')

### Figure 2B (alternative) — model recovery

`scripts/26_figure2_recovery.py`. From the script's header:

```text
26_figure2_recovery.py — Figure 2B: model recovery (50 x 50 mm).

Confusion matrices from the recovery simulation in 06_model_comparison.py (--recovery): data generated
from each model with each participant's own fitted parameters, betas and noise, then passed through
the full pipeline (nested leave-one-block-out CV -> random-effects BMS). Cell = % of simulations
(20 per generating model) in which each model had the highest protected exceedance probability.
Left: early window (80–140 ms); right: late window (300–700 ms). M4/M5 faded as in panel D.

Input : derivatives/model_comparison/recovery.tsv
Output: figures/Figure2_recovery.pdf / .svg / .png

Usage:  python scripts/26_figure2_recovery.py
```

In [ ]:
# ---- 26_figure2_recovery.py (inline; rebuild the notebook after editing the script)
def make_figure2_recovery():
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns

    # ----------------------------------------------------------------------------- CONFIG
    ROOT = PROJECT_ROOT
    REC = ROOT / "derivatives" / "model_comparison" / "recovery.tsv"
    FIG_DIR = ROOT / "figures"
    SIZE_MM = (50.0, 50.0)
    MODELS = ["M0 categories", "M1 adaptation", "M2 local", "M3 global", "M4 local+global", "M5 adapt+global"]
    SHORT = [m.split()[0] for m in MODELS]
    MCOL = {"M0": "0.35", "M1": "#55A868", "M2": "#C44E52", "M3": "#8172B3", "M4": "#937860", "M5": "#64B5CD"}
    FAINT = {"M4", "M5"}
    WINDOWS = [("local_80-140ms", "Early"), ("global_300-700ms", "Late")]
    TEXT_RC = {"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
               "font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "Helvetica", "DejaVu Sans"],
               "font.size": 6, "axes.titlesize": 6.5, "axes.labelsize": 6, "xtick.labelsize": 5.5, "ytick.labelsize": 5.5,
               "axes.linewidth": 0.5, "xtick.major.pad": 1, "ytick.major.pad": 1, "axes.labelpad": 1.5}
    # ------------------------------------------------------------------------------------


    def colour_ticks(labels):
        for tl in labels:
            m = tl.get_text()
            tl.set_color(MCOL[m])
            tl.set_alpha(0.45 if m in FAINT else 1.0)


    def main():
        d = pd.read_csv(REC, sep="\t")
        sns.set_theme(style="white", rc=TEXT_RC)
        mm = 1 / 25.4
        fig, axes = plt.subplots(1, 2, figsize=(SIZE_MM[0] * mm, SIZE_MM[1] * mm),
                                 gridspec_kw=dict(left=0.14, right=0.99, bottom=0.2, top=0.86, wspace=0.1))
        for k, (ax, (comp, lab)) in enumerate(zip(axes, WINDOWS)):
            g = d[d["component"] == comp]
            M = pd.crosstab(g["generating"], g["winner_pxp"], normalize="index").reindex(
                index=MODELS, columns=MODELS, fill_value=0).to_numpy() * 100
            ax.imshow(M, cmap="Greys", vmin=0, vmax=100, aspect="auto")
            for i in range(6):
                for j in range(6):
                    if M[i, j] >= 5:
                        ax.text(j, i, f"{M[i, j]:.0f}", ha="center", va="center", fontsize=4.6,
                                color="white" if M[i, j] > 55 else "0.15")
                ax.add_patch(plt.Rectangle((i - 0.5, i - 0.5), 1, 1, fill=False, ec=MCOL[SHORT[i]], lw=0.8))
            ax.set_xticks(range(6), SHORT, rotation=90)
            ax.set_yticks(range(6), SHORT if k == 0 else [])
            colour_ticks(ax.get_xticklabels())
            if k == 0:
                colour_ticks(ax.get_yticklabels())
                ax.set_ylabel("Generating model")
            ax.tick_params(length=0)
            for sp in ax.spines.values():
                sp.set_visible(False)
            ax.set_title(lab, pad=2)
        fig.text(0.56, 0.035, "Recovered model (% of simulations)", ha="center", fontsize=6)
        fig.text(0.565, 0.985, "Model recovery", ha="center", va="top", fontsize=6.5)

        FIG_DIR.mkdir(exist_ok=True)
        fig.savefig(FIG_DIR / "Figure2_recovery.pdf")
        fig.savefig(FIG_DIR / "Figure2_recovery.svg")
        svg = FIG_DIR / "Figure2_recovery.svg"           # SVG names fonts: Liberation Sans -> Arial (same metrics)
        svg.write_text(svg.read_text().replace("Liberation Sans", "Arial"))
        fig.savefig(FIG_DIR / "Figure2_recovery.png", dpi=600)
        print(f"Figure -> {FIG_DIR / 'Figure2_recovery'}.pdf/.svg/.png")

    return main()


run_figure('Figure 2 recovery', make_figure2_recovery)

plt.close('all')
for f in ['Figure2_recovery.png']:
    p = PROJECT_ROOT / 'figures' / f
    display(Image(filename=str(p), width=900) if p.exists() else f'missing: {p}')

## 3. Figure 3 — the local effect originates in left auditory cortex

eLORETA on a template head (MNE "sample": FreeSurfer anatomy, 3-layer BEM, oct-6 source space; standard 10-05
positions coregistered by fiducials + ICP), loose orientation 0.2, depth 0.8, λ² = 1/9, per-participant noise
covariance from the silent pre-sequence period. Statistics use the **signed** normal-orientation difference
(zero expectation under the null, unaffected by unequal trial counts). Cluster test on the 80–140 ms window mean
(cluster-forming p < .01, 2,000 sign flips, source-space adjacency).

In [ ]:
WIN_SRC, LAMBDA2, LOOSE, DEPTH = (0.08, 0.14), 1 / 9, 0.2, 0.8
AC = ["transversetemporal", "superiortemporal"]

def src_info():
    inf = mne.io.read_info(ERP_DIR / "info.fif")
    inf = mne.pick_info(inf, mne.pick_types(inf, eeg=True))
    with inf._unlock():
        inf["projs"] = []
    raw0 = mne.io.RawArray(np.zeros((len(inf["ch_names"]), 1)), inf)
    raw0.set_eeg_reference("average", projection=True)
    return raw0.info

def make_forward(inf):
    '''Coregister standard electrode positions to the template head (fiducials, then ICP) and build the
    BEM forward model (as 11/19).'''
    coreg = mne.coreg.Coregistration(inf, "sample", subjects_dir=SUBJECTS_DIR, fiducials="auto")
    coreg.fit_fiducials(verbose=False); coreg.fit_icp(n_iterations=40, nasion_weight=2.0, verbose=False)
    coreg.omit_head_shape_points(distance=10 / 1000); coreg.fit_icp(n_iterations=40, nasion_weight=2.0, verbose=False)
    tb = SUBJECTS_DIR / "sample" / "bem"
    return mne.make_forward_solution(inf, trans=coreg.trans, src=mne.read_source_spaces(tb / "sample-oct-6-src.fif"),
                                     bem=mne.read_bem_solution(tb / "sample-1280-1280-1280-bem-sol.fif"),
                                     eeg=True, meg=False, mindist=5.0, n_jobs=1)

def signed_stcs(inf, fwd):
    '''Per-participant signed eLORETA estimate of LD - LS (linear, so = difference of the estimates).'''
    out = []
    for s in subs:
        z = np.load(ERP_DIR / f"{s}.npz")
        inv = mne.minimum_norm.make_inverse_operator(inf, fwd, mne.read_cov(SRC_DIR / f"{s}-cov.fif"), loose=LOOSE, depth=DEPTH)
        ev = mne.EvokedArray((z["topo__LD"] - z["topo__LS"]) * 1e-6, inf, tmin=z["times"][0])
        out.append(mne.minimum_norm.apply_inverse(ev, inv, LAMBDA2, method="eLORETA", pick_ori="normal",
                                                  method_params=dict(max_iter=60)))
    return out

have_template = (SUBJECTS_DIR / "sample" / "bem" / "sample-oct-6-src.fif").exists()
print("template head available:", have_template)
need_stcs = RECOMPUTE_SOURCE or RUN_SPREAD_CHECK
if have_template and need_stcs:
    t0 = time.time()
    inf = src_info(); fwd = make_forward(inf); src = fwd["src"]
    stcs = signed_stcs(inf, fwd)
    labels = {lb.name: lb for lb in mne.read_labels_from_annot("sample", "aparc", subjects_dir=SUBJECTS_DIR)}
    print(f"forward + 15 inverse solutions: {time.time()-t0:.0f} s")

In [ ]:
if RECOMPUTE_SOURCE and have_template:
    X = np.stack([s.copy().crop(*WIN_SRC).data.mean(1) for s in stcs])
    T, clusters, pv, _ = mne.stats.permutation_cluster_1samp_test(
        X, threshold=stats.t.ppf(1 - 0.01 / 2, n - 1), n_permutations=N_PERM, adjacency=mne.spatial_src_adjacency(src),
        tail=0, seed=SEED, out_type="mask", n_jobs=1, verbose=False)
    mask = np.zeros_like(T, bool)
    for c, p in zip(clusters, pv):
        if p < 0.05: mask |= c
    vert = [s["vertno"] for s in src]; cluster_p = np.sort(pv)
else:
    z = np.load(SRC_DIR / "local_tmap.npz")          # saved by 19_figure3.py (same settings)
    T, mask, vert, cluster_p = z["T"], z["mask"], [z["vert_lh"], z["vert_rh"]], z["cluster_p"]
print(f"local effect: cluster p = {np.round(cluster_p[:3], 4)}, {mask.sum()} significant sources, peak |t| = {np.abs(T).max():.1f}")
report(3, "eLORETA cluster p", cluster_p[0]); report(3, "eLORETA n sources", int(mask.sum())); report(3, "eLORETA peak |t|", np.abs(T).max())

if have_template:
    labs = mne.read_labels_from_annot("sample", "aparc", subjects_dir=SUBJECTS_DIR)
    rows = []
    for lb in labs:
        if "unknown" in lb.name: continue
        h = 0 if lb.hemi == "lh" else 1
        idx = np.where(np.isin(vert[h], lb.vertices))[0] + (0 if h == 0 else len(vert[0]))
        if len(idx): rows.append(dict(label=lb.name, frac_sig=mask[idx].mean(), mean_abs_t=np.abs(T[idx]).mean()))
    inv_tab = pd.DataFrame(rows).sort_values("mean_abs_t", ascending=False)
    print(inv_tab[inv_tab.frac_sig > 0].head(12).round(2).to_string(index=False))
    for l in ["transversetemporal-lh", "superiortemporal-lh", "superiortemporal-rh", "insula-lh", "parahippocampal-lh"]:
        report(3, f"{l} fraction significant", float(inv_tab.set_index("label").loc[l, "frac_sig"]))

### 3.1 Auditory-cortex source time course (Figure 3B)

Sign-flip-aligned mean over the aparc labels transverse temporal + superior temporal (`mean_flip`), per
hemisphere; window-mean t-test. Source polarity is arbitrary (plotted with the MMN negative).

In [ ]:
if have_template and need_stcs:
    ac = {h: sum((labels[f"{nm}-{h}"] for nm in AC[1:]), labels[f"{AC[0]}-{h}"]) for h in ("lh", "rh")}
    tc = {h: np.stack([s.extract_label_time_course(ac[h], src, mode="mean_flip")[0] for s in stcs]) for h in ac}
    tsrc = stcs[0].times
else:
    z = np.load(SRC_DIR / "local_tmap.npz"); tc = {"lh": z["ac_lh"], "rh": z["ac_rh"]}; tsrc = z["times"]
w = (tsrc >= WIN_SRC[0]) & (tsrc <= WIN_SRC[1])
flip = -1.0 if tc["lh"].mean(0)[w].mean() > 0 else 1.0
for h in ["lh", "rh"]:
    X = flip * tc[h]; m = X[:, w].mean(1); t = stats.ttest_1samp(m, 0)
    sel = (tsrc >= 0.05) & (tsrc <= 0.2); pk = tsrc[sel][np.argmin(X.mean(0)[sel])] * 1e3
    print(f"auditory cortex {h}: t(14) = {t.statistic:.2f}, p = {t.pvalue:.2g}, {int((np.sign(m)==np.sign(m.mean())).sum())}/15, peak {pk:.0f} ms")
    report(3, f"auditory cortex {h} t", t.statistic)

### 3.2 Is the insula / inferior-frontal part of the cluster a separate generator?

Two checks (as `20_frontal_generator_check.py`): (i) **timing** — a second (frontal) MMN generator should peak
*after* auditory cortex; jackknife test of the peak-latency difference (Miller et al., 1998). (ii) **leakage** — a
source confined to left auditory cortex is projected through the forward model and inverted with each participant's
operator; the insula/IFG amplitude relative to auditory cortex is what spread alone produces.

In [ ]:
if have_template and RUN_SPREAD_CHECK:
    ifg = labels["insula-lh"] + labels["parsopercularis-lh"]; acl = ac["lh"]
    sa = lambda X: X * np.sign(X.mean(0)[(tsrc >= .08) & (tsrc <= .14)].mean())
    A_ = sa(tc["lh"]); I_ = sa(np.stack([s.extract_label_time_course(ifg, src, mode="mean_flip")[0] for s in stcs]))
    wpk = (tsrc >= 0.06) & (tsrc <= 0.16); tw = tsrc[wpk]
    pk = lambda X: tw[np.argmax(X.mean(0)[wpk])]
    dfull = pk(I_) - pk(A_)
    jk = np.array([pk(np.delete(I_, i, 0)) - pk(np.delete(A_, i, 0)) for i in range(n)])
    se = np.sqrt((n - 1) / n * ((jk - jk.mean()) ** 2).sum()); tj = dfull / se
    print(f"window t (insula+IFG, left): {stats.ttest_1samp(I_[:, w].mean(1), 0).statistic:.2f}")
    print(f"peak latency: auditory cortex {pk(A_)*1e3:.0f} ms, insula/IFG {pk(I_)*1e3:.0f} ms; difference {dfull*1e3:+.0f} ms, "
          f"jackknife t({n-1}) = {tj:.2f}, p = {2*stats.t.sf(abs(tj), n-1):.2f}")
    ratio_obs = I_.mean(0)[wpk].max() / A_.mean(0)[wpk].max()
    # leakage simulation: unit, sign-aligned currents in left auditory cortex only
    fwd_fixed = mne.convert_forward_solution(fwd, surf_ori=True, force_fixed=True, use_cps=True)
    idx = np.where(np.isin(vert[0], acl.vertices))[0]
    x = np.zeros(sum(len(v) for v in vert)); x[idx] = 1e-9 * mne.label_sign_flip(acl, src)
    data = fwd_fixed["sol"]["data"] @ x
    rat = []
    for s in subs:
        inv = mne.minimum_norm.make_inverse_operator(inf, fwd, mne.read_cov(SRC_DIR / f"{s}-cov.fif"), loose=LOOSE, depth=DEPTH)
        ev = mne.EvokedArray(np.tile(data[:, None], (1, 3)), inf, tmin=0); ev.set_eeg_reference("average", projection=True)
        e = mne.minimum_norm.apply_inverse(ev, inv, LAMBDA2, method="eLORETA", pick_ori="normal", method_params=dict(max_iter=60))
        rat.append(abs(e.extract_label_time_course(ifg, src, mode="mean_flip")[0][0]) / abs(e.extract_label_time_course(acl, src, mode="mean_flip")[0][0]))
    print(f"insula/IFG amplitude relative to auditory cortex: observed {100*ratio_obs:.0f}%, leakage alone {100*np.mean(rat):.0f}% "
          f"(range {100*min(rat):.0f}-{100*max(rat):.0f}%)")
    report(3, "frontal latency jackknife t", tj); report(3, "leakage ratio (%)", 100 * np.mean(rat))

### 3.3 Global effect: no reliable focal source (summary from `11_source_eloreta.py`)

The 300–700 ms GD − GS signed difference was tested the same way in 11 (with sLORETA as a check). Saved summary:

In [ ]:
gs = json.loads((SRC_DIR / "group_summary.json").read_text())["global"]
print(f"global effect: cluster p = {gs['cluster_p'][:3]}, {gs['n_sig_sources']} significant sources; "
      f"peak of mean |current|: {gs['peak_label_of_mean_abs_current']}")
print(pd.DataFrame(gs["top_labels_by_mean_abs_t"]).to_string(index=False))
report(3, "global n sources", gs["n_sig_sources"])

### Figure 3 — auditory-cortex localisation of the local effect

`scripts/19_figure3.py`. From the script's header:

```text
19_figure3.py — Figure 3 (single column, 84 x 74 mm): the local effect localises to left auditory cortex.

  A  eLORETA group t-map of the local effect (LD − LS, 80–140 ms window mean), shown only within
     significant clusters (one-sample cluster permutation on the signed source difference,
     cluster-forming p < .01, 2000 permutations); |t| because the sign depends on cortical orientation.
     Left and right lateral views of the template (inflated) surface; auditory cortex outlined.
  B  Time course of the signed source difference (LD − LS) in auditory cortex (transverse + superior
     temporal gyrus; aparc), left and right: eLORETA normal orientation, sign-flip-aligned label mean
     ("mean_flip"), mean ± SE, a-priori window shaded; window-mean t-test per hemisphere. A signed
     difference has zero expectation under the null, so — unlike source strength — it is not biased
     by unequal trial counts.

Methods identical to 11_source_eloreta.py (template "sample" head, 3-layer BEM, standard electrode
positions, loose 0.2, depth 0.8, lambda² = 1/9, per-subject noise covariance).
Inputs: derivatives/header1_erps/<sub>.npz + info.fif; derivatives/source/{<sub>-cov.fif, subjects/}
Outputs: figures/Figure3.pdf / .svg / .png; derivatives/source/figure3_stats.json;
         derivatives/source/local_tmap.npz (t-map, cluster mask, auditory-cortex time courses; cached so
         the figure can be re-styled without recomputing)
Rendering: the inflated template surface is drawn directly with matplotlib (painter's algorithm,
back-face culling, curvature shading; overlay morphed to the full-resolution surface), because the
MNE matplotlib 3D backend clips the lateral view.

Usage:  python scripts/19_figure3.py            (computes the t-map if local_tmap.npz is missing)
```

In [ ]:
# ---- 19_figure3.py (inline; rebuild the notebook after editing the script)
def make_figure3():
    from pathlib import Path
    import json

    import numpy as np
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy import stats

    # ----------------------------------------------------------------------------- CONFIG
    ROOT = PROJECT_ROOT
    CACHE_DIR = ROOT / "derivatives" / "header1_erps"
    SRC_DIR = ROOT / "derivatives" / "source"
    SUBJECTS_DIR = SRC_DIR / "subjects"
    FIG_DIR = ROOT / "figures"
    TEMPLATE = "sample"
    SIZE_MM = (84.0, 74.0)
    WIN = (0.08, 0.14)
    LAMBDA2, LOOSE, DEPTH = 1.0 / 9.0, 0.2, 0.8
    CLUSTER_P, N_PERM, SEED = 0.01, 2000, 42
    AC_LABELS = ["transversetemporal", "superiortemporal"]
    REGION = "Auditory cortex (HG + STG)"
    HCOL = {"lh": "#C44E52", "rh": "0.6"}
    TEXT_RC = {"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
               "font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "Helvetica", "DejaVu Sans"],
               "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 6.5, "xtick.labelsize": 6, "ytick.labelsize": 6,
               "legend.fontsize": 6, "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
               "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5,
               "axes.labelpad": 2}
    # ------------------------------------------------------------------------------------


    def make_forward(info):
        """Same coregistration and BEM forward model as 11_source_eloreta.py."""
        import mne
        coreg = mne.coreg.Coregistration(info, TEMPLATE, subjects_dir=SUBJECTS_DIR, fiducials="auto")
        coreg.fit_fiducials(verbose=False)
        coreg.fit_icp(n_iterations=40, nasion_weight=2.0, verbose=False)
        coreg.omit_head_shape_points(distance=10 / 1000)
        coreg.fit_icp(n_iterations=40, nasion_weight=2.0, verbose=False)
        tb = SUBJECTS_DIR / TEMPLATE / "bem"
        src = mne.read_source_spaces(tb / f"{TEMPLATE}-oct-6-src.fif")
        bem = mne.read_bem_solution(tb / f"{TEMPLATE}-1280-1280-1280-bem-sol.fif")
        return mne.make_forward_solution(info, trans=coreg.trans, src=src, bem=bem, eeg=True, meg=False,
                                         mindist=5.0, n_jobs=1)


    def local_tmap():
        """Group t-map + cluster mask of the local effect (signed eLORETA, window mean); cached."""
        out = SRC_DIR / "local_tmap.npz"
        if out.exists():
            z = np.load(out)
            return (z["T"], z["mask"], [z["vert_lh"], z["vert_rh"]], z["cluster_p"],
                    dict(lh=z["ac_lh"], rh=z["ac_rh"]), z["times"])
        import mne
        mne.set_log_level("ERROR")
        subs = sorted(f.stem for f in CACHE_DIR.glob("sub-*.npz"))
        info = mne.io.read_info(CACHE_DIR / "info.fif")
        info = mne.pick_info(info, mne.pick_types(info, eeg=True))
        with info._unlock():
            info["projs"] = []
        raw0 = mne.io.RawArray(np.zeros((len(info["ch_names"]), 1)), info)
        raw0.set_eeg_reference("average", projection=True)
        info = raw0.info
        fwd = make_forward(info)
        src = fwd["src"]
        labs = {lb.name: lb for lb in mne.read_labels_from_annot(TEMPLATE, "aparc", subjects_dir=SUBJECTS_DIR)}
        ac = {h: sum((labs[f"{n}-{h}"] for n in AC_LABELS[1:]), labs[f"{AC_LABELS[0]}-{h}"]) for h in ("lh", "rh")}
        X, tcs = [], {"lh": [], "rh": []}
        for sub in subs:
            z = np.load(CACHE_DIR / f"{sub}.npz")
            cov = mne.read_cov(SRC_DIR / f"{sub}-cov.fif")
            inv = mne.minimum_norm.make_inverse_operator(info, fwd, cov, loose=LOOSE, depth=DEPTH)
            ev = mne.EvokedArray((z["topo__LD"] - z["topo__LS"]) * 1e-6, info, tmin=z["times"][0])
            stc = mne.minimum_norm.apply_inverse(ev, inv, LAMBDA2, method="eLORETA", pick_ori="normal",
                                                 method_params=dict(max_iter=60))
            for h in ("lh", "rh"):
                tcs[h].append(stc.extract_label_time_course(ac[h], src, mode="mean_flip")[0])
            times = stc.times
            X.append(stc.copy().crop(*WIN).data.mean(1))
            print(f"{sub}: done", flush=True)
        X = np.stack(X)
        T, clusters, pv, _ = mne.stats.permutation_cluster_1samp_test(
            X, threshold=stats.t.ppf(1 - CLUSTER_P / 2, len(X) - 1), n_permutations=N_PERM,
            adjacency=mne.spatial_src_adjacency(src), tail=0, seed=SEED, out_type="mask", n_jobs=1, verbose=False)
        mask = np.zeros_like(T, bool)
        for c, p in zip(clusters, pv):
            if p < 0.05:
                mask |= c
        vert = [s["vertno"] for s in src]
        tcs = {h: np.stack(v) for h, v in tcs.items()}
        np.savez_compressed(out, T=T, mask=mask, vert_lh=vert[0], vert_rh=vert[1], cluster_p=np.sort(pv),
                            ac_lh=tcs["lh"], ac_rh=tcs["rh"], times=times)
        return T, mask, vert, np.sort(pv), tcs, times


    def render(ax, values, vert, hemi, lim, cmap):
        """Lateral view of the inflated surface: back-face culling + depth sort, curvature shading,
        overlay (|t| within significant clusters) morphed to all vertices; auditory cortex outlined."""
        import mne
        import nibabel
        from matplotlib.collections import PolyCollection, LineCollection
        sd = SUBJECTS_DIR / TEMPLATE
        rr, tris = mne.read_surface(sd / "surf" / f"{hemi}.inflated")
        curv = nibabel.freesurfer.read_morph_data(sd / "surf" / f"{hemi}.curv")
        stc = mne.SourceEstimate(values[:, None], vertices=vert, tmin=0, tstep=1, subject=TEMPLATE)
        full = mne.compute_source_morph(stc, TEMPLATE, TEMPLATE, subjects_dir=SUBJECTS_DIR, spacing=None,
                                        smooth=5, verbose=False).apply(stc)
        val = full.data[: len(full.vertices[0]), 0] if hemi == "lh" else full.data[len(full.vertices[0]):, 0]
        vfull = np.zeros(len(rr))
        vfull[full.vertices[0 if hemi == "lh" else 1]] = val
        sgn = -1 if hemi == "lh" else 1                     # camera on the lateral side
        tri = rr[tris]
        nrm = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
        vis = sgn * nrm[:, 0] > 0
        order = np.argsort(sgn * tri[:, :, 0].mean(1)[vis])  # far (medial) first
        fi = np.where(vis)[0][order]
        u = (-rr[:, 1] if hemi == "lh" else rr[:, 1])          # anterior to the left for lh, right for rh
        xy = np.column_stack([u, rr[:, 2]])
        base = np.where(curv[tris[fi]].mean(1) > 0, 0.55, 0.78)
        col = np.repeat(base[:, None], 3, 1)
        fv = vfull[tris[fi]].mean(1)
        on = fv > lim * 0.02
        over = cmap(np.clip(fv / lim, 0, 1))[:, :3]
        a = np.clip(fv / (lim * 0.25), 0, 1)[:, None]
        col = np.where(on[:, None], (1 - a) * col + a * over, col)
        ax.add_collection(PolyCollection(xy[tris[fi]], facecolors=col, edgecolors="none", antialiased=False,
                                         rasterized=True))
        # auditory-cortex outline: boundary edges of the label's visible faces
        labs = mne.read_labels_from_annot(TEMPLATE, "aparc", hemi=hemi, subjects_dir=SUBJECTS_DIR, verbose=False)
        inlab = np.zeros(len(rr), bool)
        for lb in labs:
            if lb.name.split("-")[0] in AC_LABELS:
                inlab[lb.vertices] = True
        lf = tris[vis][inlab[tris[vis]].all(1)]
        e = np.sort(np.concatenate([lf[:, [0, 1]], lf[:, [1, 2]], lf[:, [2, 0]]]), axis=1)
        ue, cnt = np.unique(e, axis=0, return_counts=True)
        ax.add_collection(LineCollection(xy[ue[cnt == 1]], colors="k", linewidths=0.5))
        ax.set_xlim(xy[:, 0].min(), xy[:, 0].max())
        ax.set_ylim(xy[:, 1].min(), xy[:, 1].max())
        ax.set_aspect("equal")
        ax.set_axis_off()


    def main():
        import mne
        mne.set_log_level("ERROR")
        T, mask, vert, cluster_p, tcs, times = local_tmap()
        absT = np.abs(T) * mask
        lim = float(np.ceil(np.abs(T).max()))
        cmap = plt.get_cmap("YlOrRd")
        n = len(tcs["lh"])

        sns.set_theme(style="ticks", rc=TEXT_RC)
        mm = 1 / 25.4
        fig = plt.figure(figsize=(SIZE_MM[0] * mm, SIZE_MM[1] * mm))
        out = dict(cluster_p=[float(p) for p in cluster_p[:3]], n_sig_sources=int(mask.sum()),
                   t_max=float(np.abs(T).max()))
        # ---------------- A: brains
        for k, h in enumerate(("lh", "rh")):
            ax = fig.add_axes([0.03 + k * 0.42, 0.57, 0.40, 0.33])
            render(ax, absT, vert, h, lim, cmap)
            ax.set_title("Left" if h == "lh" else "Right", pad=1)
        cax = fig.add_axes([0.89, 0.62, 0.022, 0.22])
        cb = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, lim)), cax=cax)
        cb.set_label("|t|", labelpad=1)
        cb.ax.tick_params(labelsize=5.5)
        fig.text(0.02, 0.99, "A", fontsize=9, weight="bold", va="top")
        fig.text(0.09, 0.985, f"Local effect (LD − LS), {int(WIN[0] * 1e3)}–{int(WIN[1] * 1e3)} ms", fontsize=7, va="top")

        # ---------------- B: signed source difference in auditory cortex
        ax = fig.add_axes([0.17, 0.135, 0.79, 0.36])
        ax.axvspan(WIN[0] * 1e3, WIN[1] * 1e3, color="0.92", lw=0, zorder=0)
        t = times * 1e3
        w = (times >= WIN[0]) & (times <= WIN[1])
        flip = -1.0 if tcs["lh"].mean(0)[w].mean() > 0 else 1.0   # mean_flip sign is arbitrary: plot MMN as negative
        scale = np.abs(tcs["lh"].mean(0)).max()
        for h, lab in (("lh", "left"), ("rh", "right")):
            X = flip * tcs[h] / scale
            m, se = X.mean(0), X.std(0, ddof=1) / np.sqrt(n)
            wm = X[:, w].mean(1)
            r = stats.ttest_1samp(wm, 0)
            same = int((np.sign(wm) == np.sign(wm.mean())).sum())
            ptxt = "p < .001" if r.pvalue < 0.001 else f"p = {r.pvalue:.3f}".replace("0.", ".")
            ax.plot(t, m, color=HCOL[h], lw=1.1, label=f"{lab}  t({n - 1}) = {r.statistic:.1f}, {ptxt}".replace("-", "−"))
            ax.fill_between(t, m - se, m + se, color=HCOL[h], alpha=0.25, lw=0)
            pk = t[(t >= 50) & (t <= 200)][np.argmin(m[(t >= 50) & (t <= 200)])]
            out[f"auditory_cortex_{h}"] = dict(t=float(r.statistic), p=float(r.pvalue), n_same_sign=same,
                                               group_peak_ms=float(pk))
        ax.axvline(0, color="k", lw=0.5)
        ax.axhline(0, color="k", lw=0.4)
        ax.set_xlim(-100, 500)
        ax.set_xlabel("Time from 5th sound (ms)")
        ax.set_ylabel("LD − LS source current\n(a.u., normalised)")
        ax.legend(frameon=False, loc="lower right", handlelength=1.2, borderaxespad=0.1, title="Auditory cortex",
                  title_fontsize=6)
        sns.despine(ax=ax)
        fig.text(0.02, 0.55, "B", fontsize=9, weight="bold", va="top")

        FIG_DIR.mkdir(exist_ok=True)
        fig.savefig(FIG_DIR / "Figure3.pdf", dpi=600)
        fig.savefig(FIG_DIR / "Figure3.svg", dpi=600)
        svg = FIG_DIR / "Figure3.svg"                     # SVG names fonts: Liberation Sans -> Arial (same metrics)
        svg.write_text(svg.read_text().replace("Liberation Sans", "Arial"))
        fig.savefig(FIG_DIR / "Figure3.png", dpi=600)
        (SRC_DIR / "figure3_stats.json").write_text(json.dumps(out, indent=2))
        print(json.dumps(out, indent=2))

    return main()


run_figure('Figure 3', make_figure3)

plt.close('all')
for f in ['Figure3.png']:
    p = PROJECT_ROOT / 'figures' / f
    display(Image(filename=str(p), width=900) if p.exists() else f'missing: {p}')

## 4. DCM — effective connectivity (results read from SPM)

The DCMs are fitted in MATLAB/SPM12 and cannot run here. Run order (from `scripts/dcm/`):
`dcm_01_prepare.m` → `dcm_02_fit.m` → `dcm_03_peb.m` → `dcm_04_robustness.m` → (after
`python scripts/25_dcm_surprise_export.py`) `dcm_05_surprise.m` (0–600 ms) → `dcm_06_early.m` (0–250 ms) →
`dcm_07_search_full.m`. This section reads their outputs.

**Codings.** *Surprise*: each participant's condition-mean learner surprise (M2 local, M3 global), relative to
LS/GS, scaled to a maximum of 1. *Interaction*: 0/1 labels plus a local × global term. **Credible** = posterior
probability of being present > .95 after per-connection Bayesian model reduction (`Pp_search`; falls back to
the sign probability `Pp` where that column is missing). **Robust** = credible with the same sign in both codings.

In [ ]:
# ---------------------------------------------------------------- DCM: connection tables per window x coding
DCM_DIR = ROOT / "derivatives" / "dcm"
dcm_tabs = []
for win, sub in [("0–600 ms", "surprise_coded"), ("0–250 ms", "early_window")]:
    for cod in ["surprise", "interaction"]:
        f = DCM_DIR / sub / cod / "connections.csv"
        if not f.exists():
            print("missing:", f); continue
        d = pd.read_csv(f)
        pp = "Pp_search" if "Pp_search" in d else "Pp"
        ep = "Ep_search" if "Ep_search" in d else "Ep"
        dcm_tabs.append(d.assign(window=win, coding=cod, Ep_use=d[ep], credible=d[pp] > 0.95, stat=pp))
dcm = pd.concat(dcm_tabs, ignore_index=True)
print("credibility statistic used:", dcm.groupby(["window", "coding"])["stat"].first().to_dict())

# robust = credible with the same sign in both codings
rob = (dcm[dcm.effect.isin(["local", "global"])]
       .groupby(["window", "effect", "connection"])
       .apply(lambda g: pd.Series(dict(robust=bool(g.credible.all() and np.sign(g.Ep_use).nunique() == 1),
                                       mean_log_gain=g.Ep_use.mean(),
                                       **{f"Ep_{c}": float(g.loc[g.coding == c, "Ep_use"].iloc[0]) for c in g.coding})),
              include_groups=False)
       .reset_index())
for (win, eff), g in rob[rob.robust].groupby(["window", "effect"], sort=False):
    print(f"\n{win}, {eff}: robust connection changes (log gain)")
    print(g[["connection", "mean_log_gain", "Ep_surprise", "Ep_interaction"]].round(2).to_string(index=False))

### DCM figure — full response (0–600 ms) vs MMN window (0–250 ms)

`scripts/28_figure_dcm_windows.py`. From the script's header:

```text
28_figure_dcm_windows.py — DCM figure: full response (0–600 ms) vs MMN window (0–250 ms), 173.695 x 80 mm.

Question: does global deviance really modulate the auditory forward connection (A1 -> STG), or is that
forced by the model? The only driving input enters at A1, so over 0–600 ms the late P3b of global
deviants has to pass through A1 -> STG. Fitting 0–250 ms removes the P3b.

Each window was fitted with two codings of the between-condition effects (dcm_05_surprise.m, 0–600 ms;
dcm_06_early.m, 0–250 ms):
  surprise     local / global = each participant's condition-mean learner surprise (M2 / M3)
  interaction  0/1 labels + local x global term (global = global effect on local standards only)

Credible = posterior probability of being present > .95 after per-connection Bayesian model reduction
(Pp_search). If a CSV has no Pp_search column (dcm_05 output before dcm_07_search_full.m was run), the
sign probability of the family-level BMA (Pp_sign) is used instead and a warning is printed.
Robust = credible with the same sign in BOTH codings.

  A–D  Networks: rows local / global deviance, columns 0–600 / 0–250 ms. Coloured arrows = robust
       (orange stronger, blue weaker than for standards; width ~ mean |log gain| over codings);
       thin grey = credible in only one coding or with opposite signs; absent = not credible.
  E    Every connection credible in at least one coding x window: log gain per coding x window,
       credible estimates outlined.
  F    Fit: variance explained per participant (skipped if the DCM .mat files are not present).

Inputs : derivatives/dcm/surprise_coded/<coding>/connections.csv (0–600 ms),
         derivatives/dcm/early_window/<coding>/connections.csv (0–250 ms), and their DCM_sub-*.mat
Outputs: figures/Figure_DCM_windows.pdf / .svg / .png; derivatives/dcm/early_window/window_comparison.csv

Usage:  python scripts/28_figure_dcm_windows.py
```

In [ ]:
# ---- 28_figure_dcm_windows.py (inline; rebuild the notebook after editing the script)
def make_figure_dcm_windows():
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
    from matplotlib.colors import TwoSlopeNorm
    import seaborn as sns
    from scipy.io import loadmat

    # ----------------------------------------------------------------------------- CONFIG
    ROOT = PROJECT_ROOT
    DCM_DIR = ROOT / "derivatives" / "dcm"
    FIG_DIR = ROOT / "figures"
    SIZE_MM = (173.695, 80.0)
    WINDOWS = {"full": ("surprise_coded", "0–600 ms"), "early": ("early_window", "0–250 ms")}
    CODINGS = {"surprise": "Surprise", "interaction": "Interaction"}
    EFFECTS = ["local", "global"]
    PP_THR = 0.95
    SRC = ["lA1", "rA1", "lSTG", "rSTG", "lIFG", "rIFG", "lIPL", "rIPL"]
    POS = {"lA1": (-1.0, 0.0), "lSTG": (-1.0, 1.0), "lIFG": (-1.55, 2.05), "lIPL": (-0.45, 2.05),
           "rA1": (1.0, 0.0), "rSTG": (1.0, 1.0), "rIFG": (1.55, 2.05), "rIPL": (0.45, 2.05)}
    FWD = [("lA1", "lSTG"), ("rA1", "rSTG"), ("lSTG", "lIFG"), ("rSTG", "rIFG"), ("lSTG", "lIPL"), ("rSTG", "rIPL")]
    LAT = [("lIFG", "lIPL"), ("rIFG", "rIPL")]
    C_UP, C_DOWN, C_NS = "#D55E00", "#0072B2", "0.72"
    C_EFF = {"local": "#C44E52", "global": "#8172B3"}
    TEXT_RC = {"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
               "font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "Helvetica", "DejaVu Sans"],
               "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 6.5, "xtick.labelsize": 6, "ytick.labelsize": 6,
               "legend.fontsize": 6, "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
               "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5,
               "axes.labelpad": 2}
    # ------------------------------------------------------------------------------------


    def load_all():
        """One long table: window, coding, effect, connection, Ep, credible."""
        out = []
        for win, (sub, _) in WINDOWS.items():
            for cod in CODINGS:
                d = pd.read_csv(DCM_DIR / sub / cod / "connections.csv")
                if "Pp_search" in d:
                    ep, pp = d["Ep_search"], d["Pp_search"]
                else:
                    print(f"WARNING: {sub}/{cod} has no per-connection BMR; using Pp_sign "
                          f"(run scripts/dcm/dcm_07_search_full.m)")
                    ep, pp = d["Ep"], d["Pp"]
                out.append(pd.DataFrame(dict(window=win, coding=cod, effect=d["effect"], connection=d["connection"],
                                             Ep=ep, credible=pp > PP_THR)))
        d = pd.concat(out, ignore_index=True)
        return d[d["effect"].isin(EFFECTS)]


    def robustness(d):
        """Per window x effect x connection: robust (credible, same sign, both codings) and mean log gain."""
        rows = []
        for (win, eff, conn), g in d.groupby(["window", "effect", "connection"]):
            g = g.set_index("coding").reindex(list(CODINGS))
            cred = g["credible"].astype(bool)
            same = np.sign(g["Ep"]).nunique() == 1
            rows.append(dict(window=win, effect=eff, connection=conn, n_credible=int(cred.sum()),
                             robust=bool(cred.all() and same), sign=int(np.sign(g["Ep"].mean())),
                             Ep_mean=float(g["Ep"].mean()),
                             **{f"Ep_{c}": float(g.loc[c, "Ep"]) for c in CODINGS},
                             **{f"cred_{c}": bool(cred[c]) for c in CODINGS}))
        return pd.DataFrame(rows)


    def r2_by_window():
        out = {}
        for win, (sub, _) in WINDOWS.items():
            for cod in CODINGS:
                vals = []
                for f in sorted((DCM_DIR / sub / cod).glob("DCM_sub-*.mat")):
                    D = loadmat(f, squeeze_me=True, struct_as_record=False, variable_names=["DCM"])["DCM"]
                    H = np.concatenate([np.ravel(h) for h in np.atleast_1d(D.H)])
                    R = np.concatenate([np.ravel(x) for x in np.atleast_1d(D.R)])
                    vals.append(1 - np.sum(R ** 2) / np.sum((H + R) ** 2))
                if vals:
                    out[(win, cod)] = np.array(vals)
        return out


    def draw_network(ax, rob, win, eff):
        ax.set_xlim(-2.05, 2.05)
        ax.set_ylim(-0.55, 2.4)
        ax.set_aspect("equal")
        ax.set_axis_off()
        for a, b in LAT:
            (x0, y0), (x1, y1) = POS[a], POS[b]
            ax.plot([x0, x1], [y0, y1], ls=(0, (2, 2)), color="0.8", lw=0.5, zorder=1)
        for side in ("l", "r"):
            x, y = POS[f"{side}A1"]
            ax.add_patch(FancyArrowPatch((x, y - 0.5), (x, y - 0.17), arrowstyle="-|>", mutation_scale=6,
                                         color="0.45", lw=0.7))
        s = rob[(rob["window"] == win) & (rob["effect"] == eff)].set_index("connection")
        for lo, hi in FWD:
            for p0, p1 in ((lo, hi), (hi, lo)):                                   # forward, then backward
                row = s.loc[f"{p0}->{p1}"]
                if row["robust"]:
                    col = C_UP if row["sign"] > 0 else C_DOWN
                    lw = 0.6 + 1.6 * min(abs(row["Ep_mean"]), 1.2)
                    z = 3
                elif row["n_credible"] > 0:
                    col, lw, z = C_NS, 0.5, 2
                else:
                    continue
                ax.add_patch(FancyArrowPatch(POS[p0], POS[p1], connectionstyle="arc3,rad=0.28", arrowstyle="-|>",
                                             mutation_scale=4 + 2 * lw, lw=lw, color=col, shrinkA=8, shrinkB=8,
                                             zorder=z))
        for name, (x, y) in POS.items():
            ax.add_patch(FancyBboxPatch((x - 0.22, y - 0.13), 0.44, 0.26, boxstyle="round,pad=0,rounding_size=0.08",
                                        fc="white", ec="0.3", lw=0.6, zorder=4))
            ax.text(x, y, name[1:], ha="center", va="center", fontsize=5.8, zorder=5)
        ax.text(-1.45, -0.5, "L", ha="center", va="bottom", fontsize=5.5, color="0.45")
        ax.text(1.45, -0.5, "R", ha="center", va="bottom", fontsize=5.5, color="0.45")


    def pretty(conn):
        a, b = conn.split("->")
        return f"{a[0].upper()} {a[1:]} → {b[1:]}"


    def main():
        d = load_all()
        rob = robustness(d)
        rob.to_csv(DCM_DIR / "early_window" / "window_comparison.csv", index=False)
        for win in WINDOWS:
            for eff in EFFECTS:
                r = rob[(rob.window == win) & (rob.effect == eff) & rob.robust]
                print(f"{WINDOWS[win][1]:9s} {eff:6s} robust: " +
                      ", ".join(f"{c} {e:+.2f}" for c, e in zip(r.connection, r.Ep_mean)))
        r2 = r2_by_window()

        sns.set_theme(style="ticks", rc=TEXT_RC)
        mm = 1 / 25.4
        fig = plt.figure(figsize=(SIZE_MM[0] * mm, SIZE_MM[1] * mm))

        # ---------------- A–D: networks (rows = effect, columns = window)
        W, H = 0.205, 0.42
        letters = iter("ABCD")
        for i, eff in enumerate(EFFECTS):
            for j, win in enumerate(WINDOWS):
                x0, y0 = 0.035 + j * (W + 0.035), 0.50 - i * (H + 0.02)
                ax = fig.add_axes([x0, y0, W, H])
                draw_network(ax, rob, win, eff)
                fig.text(x0 - 0.012, y0 + H + 0.01, next(letters), fontsize=9, weight="bold", va="top")
                if i == 0:
                    ax.set_title(WINDOWS[win][1], pad=1)
            fig.text(0.012, 0.50 - i * (H + 0.02) + H / 2, f"{eff.capitalize()} deviance", rotation=90,
                     ha="center", va="center", color=C_EFF[eff], weight="bold")
        # legend
        x = 0.04
        for col, txt in ((C_UP, "stronger"), (C_DOWN, "weaker")):
            fig.add_artist(plt.Line2D([x, x + 0.02], [0.03, 0.03], color=col, lw=1.6, transform=fig.transFigure))
            fig.text(x + 0.024, 0.03, txt, va="center", fontsize=5.8)
            x += 0.075
        fig.add_artist(plt.Line2D([x, x + 0.02], [0.03, 0.03], color=C_NS, lw=0.6, transform=fig.transFigure))
        fig.text(x + 0.024, 0.03, "one coding only", va="center", fontsize=5.8)

        # ---------------- E: estimate matrix
        cols = [(w, c) for w in WINDOWS for c in CODINGS]
        keep = rob.groupby(["effect", "connection"]).filter(lambda g: g["n_credible"].sum() > 0)
        wide = []
        for (eff, conn), g in keep.groupby(["effect", "connection"]):
            g = g.set_index("window")
            wide.append(dict(effect=eff, connection=conn,
                             **{f"{w}|{c}": g.loc[w, f"Ep_{c}"] for w, c in cols},
                             **{f"cred {w}|{c}": g.loc[w, f"cred_{c}"] for w, c in cols},
                             order=(EFFECTS.index(eff), SRC.index(conn.split("->")[0]),
                                    SRC.index(conn.split("->")[1]))))
        wide = pd.DataFrame(wide).sort_values("order").reset_index(drop=True)
        axE = fig.add_axes([0.60, 0.17, 0.17, 0.73])
        E = wide[[f"{w}|{c}" for w, c in cols]].to_numpy(float)
        lim = np.ceil(np.nanmax(np.abs(E)) * 10) / 10
        im = axE.imshow(E, cmap="RdBu_r", norm=TwoSlopeNorm(0, -lim, lim), aspect="auto")
        for i in range(len(wide)):
            for j, (w, c) in enumerate(cols):
                if wide.loc[i, f"cred {w}|{c}"]:
                    axE.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="k", lw=0.6))
        axE.set_yticks(range(len(wide)), [pretty(c) for c in wide["connection"]], fontsize=5.2)
        for tl, eff in zip(axE.get_yticklabels(), wide["effect"]):
            tl.set_color(C_EFF[eff])
        axE.set_xticks(range(len(cols)), [CODINGS[c] for _, c in cols], rotation=90, fontsize=5.5)
        for j, w in enumerate(WINDOWS):
            axE.text(1.5 * j + 0.5 * j + 0.5, -1.0, WINDOWS[w][1], ha="center", va="bottom", fontsize=5.8)
        axE.axvline(1.5, color="k", lw=0.8)
        axE.axhline(int((wide["effect"] == "local").sum()) - 0.5, color="k", lw=0.8)
        axE.tick_params(length=0)
        for sp in axE.spines.values():
            sp.set_visible(False)
        cax = fig.add_axes([0.785, 0.70, 0.01, 0.2])
        cb = fig.colorbar(im, cax=cax)
        cb.ax.tick_params(labelsize=5, length=1.5, pad=1)
        cb.set_label("log gain", fontsize=5.5, labelpad=1)
        fig.text(0.475, 0.985, "E", fontsize=9, weight="bold", va="top")

        # ---------------- F: fit
        if r2:
            axF = fig.add_axes([0.905, 0.30, 0.09, 0.60])
            for j, key in enumerate(cols):
                if key not in r2:
                    continue
                v = r2[key]
                axF.scatter(np.full(len(v), j) + np.random.default_rng(j).uniform(-0.15, 0.15, len(v)), v, s=3,
                            color="0.55", lw=0)
                axF.plot([j - 0.3, j + 0.3], [np.median(v)] * 2, color="k", lw=1)
            axF.set_xticks(range(len(cols)), [f"{WINDOWS[w][1].replace(' ms', '')} {CODINGS[c][:4]}." for w, c in cols],
                           rotation=90, fontsize=5)
            axF.set_ylim(0.75, 1.0)
            axF.set_ylabel("Variance explained")
            sns.despine(ax=axF)
            fig.text(0.85, 0.985, "F", fontsize=9, weight="bold", va="top")

        FIG_DIR.mkdir(exist_ok=True)
        for ext in ("pdf", "svg"):
            fig.savefig(FIG_DIR / f"Figure_DCM_windows.{ext}")
        svg = FIG_DIR / "Figure_DCM_windows.svg"             # SVG names fonts: Liberation Sans -> Arial (same metrics)
        svg.write_text(svg.read_text().replace("Liberation Sans", "Arial"))
        fig.savefig(FIG_DIR / "Figure_DCM_windows.png", dpi=600)
        print(f"Figure -> {FIG_DIR / 'Figure_DCM_windows'}.pdf/.svg/.png")

    return main()


run_figure('DCM windows', make_figure_dcm_windows)

plt.close('all')
for f in ['Figure_DCM_windows.png']:
    p = PROJECT_ROOT / 'figures' / f
    display(Image(filename=str(p), width=900) if p.exists() else f'missing: {p}')

### Supplementary — DCM robustness across five analyses (0/1 coding)

`scripts/24_figure_dcm_robust.py`. From the script's header:

```text
24_figure_dcm_robust.py — DCM figure built on robustness (173.695 x 78 mm).

Five analyses of the same data: main (0–600 ms, free-orientation dipoles, window mean removed) and
four variants from dcm_04_robustness.m — win400 (0–400 ms), iplsup (IPL node moved to angular gyrus),
h0 (window mean not removed), img (cortical patches with anatomically fixed orientation).
A connection change counts as supported in an analysis if its group posterior (PEB + BMR, Bayesian model
average) has Pp > .95. Support = number of analyses (of 5) in which it is credible with the majority sign.

  A/B  Local / global deviance network: coloured arrows = supported in ≥ 3 of 5 analyses (orange =
       stronger, blue = weaker than for standards), width = number of supporting analyses; grey = < 3.
  C    Every connection supported in ≥ 3 analyses for either effect: estimate per analysis
       (log gain; colour) with credible estimates outlined, and the support count.
  D    Model fit per analysis: variance explained per participant (in the DCM's spatial modes).

Inputs : derivatives/dcm/robustness/robustness_summary_corrected.csv (all five analyses, Pp recomputed
         from each BMA), derivatives/dcm/fits/DCM_*_full.mat, derivatives/dcm/robustness/<variant>/DCM_*.mat
Outputs: figures/Figure_DCM_robust.pdf / .svg / .png; derivatives/dcm/robustness/support_table.csv

Usage:  python scripts/24_figure_dcm_robust.py
```

In [ ]:
# ---- 24_figure_dcm_robust.py (inline; rebuild the notebook after editing the script)
def make_figure_dcm_robust():
    from pathlib import Path
    import json

    import numpy as np
    import pandas as pd
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
    from matplotlib.colors import TwoSlopeNorm
    import seaborn as sns
    from scipy.io import loadmat

    # ----------------------------------------------------------------------------- CONFIG
    ROOT = PROJECT_ROOT
    DCM_DIR = ROOT / "derivatives" / "dcm"
    FIG_DIR = ROOT / "figures"
    SIZE_MM = (173.695, 78.0)
    ANALYSES = ["main", "win400", "iplsup", "h0", "img"]
    ALAB = {"main": "Main", "win400": "0–400 ms", "iplsup": "IPL moved", "h0": "No mean\nremoval",
            "img": "Fixed-orient.\npatches"}
    FIT_GLOB = {"main": "fits/DCM_sub-*_full.mat", **{v: f"robustness/{v}/DCM_sub-*.mat" for v in ANALYSES[1:]}}
    PP_THR, MIN_SUPPORT = 0.95, 3
    SRC = ["lA1", "rA1", "lSTG", "rSTG", "lIFG", "rIFG", "lIPL", "rIPL"]
    LABEL = {s: s[1:] for s in SRC}
    POS = {"lA1": (-1.0, 0.0), "lSTG": (-1.0, 1.0), "lIFG": (-1.55, 2.05), "lIPL": (-0.45, 2.05),
           "rA1": (1.0, 0.0), "rSTG": (1.0, 1.0), "rIFG": (1.55, 2.05), "rIPL": (0.45, 2.05)}
    FWD = [("lA1", "lSTG"), ("rA1", "rSTG"), ("lSTG", "lIFG"), ("rSTG", "rIFG"), ("lSTG", "lIPL"), ("rSTG", "rIPL")]
    LAT = [("lIFG", "lIPL"), ("rIFG", "rIPL")]
    C_UP, C_DOWN, C_NS = "#D55E00", "#0072B2", "0.85"
    C_EFF = {"local": "#C44E52", "global": "#8172B3"}
    TEXT_RC = {"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
               "font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "Helvetica", "DejaVu Sans"],
               "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 6.5, "xtick.labelsize": 6, "ytick.labelsize": 6,
               "legend.fontsize": 6, "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
               "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5,
               "axes.labelpad": 2}
    # ------------------------------------------------------------------------------------


    def dense(x):
        return x.toarray() if hasattr(x, "toarray") else np.asarray(x, float)


    def support_table(d):
        """Per effect x connection: estimates per analysis, credible flags, majority sign, support count."""
        rows = []
        for (eff, conn), g in d.groupby(["effect", "connection"]):
            g = g.set_index("variant").reindex(ANALYSES)
            cred = g["Pp"] > PP_THR
            signs = np.sign(g["Ep"][cred])
            sign = int(np.sign(signs.sum())) if cred.any() and signs.sum() != 0 else 0
            k = int(((np.sign(g["Ep"]) == sign) & cred).sum()) if sign else 0
            rows.append(dict(effect=eff, connection=conn, sign=sign, support=k,
                             **{f"Ep_{a}": g.loc[a, "Ep"] for a in ANALYSES},
                             **{f"cred_{a}": bool(cred[a]) for a in ANALYSES}))
        return pd.DataFrame(rows)


    def r2_per_analysis():
        out = {}
        for a, pat in FIT_GLOB.items():
            vals = []
            for f in sorted(DCM_DIR.glob(pat)):
                D = loadmat(f, squeeze_me=True, struct_as_record=False)["DCM"]
                H = [dense(h) for h in np.atleast_1d(D.H)]
                R = [dense(x) for x in np.atleast_1d(D.R)]
                vals.append(1 - sum((x ** 2).sum() for x in R) / sum(((h + x) ** 2).sum() for h, x in zip(H, R)))
            out[a] = np.array(vals)
        return out


    def draw_network(ax, sup, eff):
        ax.set_xlim(-2.05, 2.05)
        ax.set_ylim(-0.55, 2.45)
        ax.set_aspect("equal")
        ax.set_axis_off()
        for a, b in LAT:
            (x0, y0), (x1, y1) = POS[a], POS[b]
            ax.plot([x0, x1], [y0, y1], ls=(0, (2, 2)), color="0.75", lw=0.6, zorder=1)
        for side in ("l", "r"):
            x, y = POS[f"{side}A1"]
            ax.add_patch(FancyArrowPatch((x, y - 0.5), (x, y - 0.2), arrowstyle="-|>", mutation_scale=6,
                                         color="0.45", lw=0.7))
        s = sup[sup["effect"] == eff].set_index("connection")
        for lo, hi in FWD:
            for p0, p1 in ((lo, hi), (hi, lo)):                                   # forward, then backward
                row = s.loc[f"{p0}->{p1}"]
                ok = row["support"] >= MIN_SUPPORT
                col = (C_UP if row["sign"] > 0 else C_DOWN) if ok else C_NS
                lw = 0.9 * row["support"] - 1.2 if ok else 0.5
                ax.add_patch(FancyArrowPatch(POS[p0], POS[p1], connectionstyle="arc3,rad=0.28", arrowstyle="-|>",
                                             mutation_scale=5 + 2 * lw, lw=lw, color=col, shrinkA=8, shrinkB=8,
                                             zorder=3 if ok else 2))
        for name, (x, y) in POS.items():
            ax.add_patch(FancyBboxPatch((x - 0.22, y - 0.13), 0.44, 0.26, boxstyle="round,pad=0,rounding_size=0.08",
                                        fc="white", ec="0.3", lw=0.7, zorder=4))
            ax.text(x, y, LABEL[name], ha="center", va="center", fontsize=6.5, zorder=5)
        ax.text(-1.0, 2.42, "Left", ha="center", va="top", fontsize=6, color="0.4")
        ax.text(1.0, 2.42, "Right", ha="center", va="top", fontsize=6, color="0.4")


    def pretty(conn):
        a, b = conn.split("->")
        return f"{a[0].upper()} {a[1:]} → {b[1:]}"


    def main():
        d = pd.read_csv(DCM_DIR / "robustness" / "robustness_summary_corrected.csv")
        sup = support_table(d)
        sup.to_csv(DCM_DIR / "robustness" / "support_table.csv", index=False)
        r2 = r2_per_analysis()

        sns.set_theme(style="ticks", rc=TEXT_RC)
        mm = 1 / 25.4
        fig = plt.figure(figsize=(SIZE_MM[0] * mm, SIZE_MM[1] * mm))
        # ---------------- A/B networks
        for k, (eff, lab) in enumerate((("local", "A"), ("global", "B"))):
            ax = fig.add_axes([0.0 + k * 0.255, 0.13, 0.25, 0.80])
            draw_network(ax, sup, eff)
            ax.set_title(f"{eff.capitalize()} deviance", color=C_EFF[eff], weight="bold", pad=0)
            fig.text(0.005 + k * 0.255, 0.985, lab, fontsize=9, weight="bold", va="top")
        # legend
        x = 0.02
        for col, txt in ((C_UP, "stronger"), (C_DOWN, "weaker")):
            fig.add_artist(plt.Line2D([x, x + 0.022], [0.06, 0.06], color=col, lw=1.6, transform=fig.transFigure))
            fig.text(x + 0.026, 0.06, txt, va="center", fontsize=6)
            x += 0.085
        for kk in (3, 4, 5):
            fig.add_artist(plt.Line2D([x, x + 0.022], [0.06, 0.06], color="0.3", lw=0.9 * kk - 1.2,
                                      transform=fig.transFigure))
            fig.text(x + 0.026, 0.06, f"{kk}/5", va="center", fontsize=6)
            x += 0.06
        fig.add_artist(plt.Line2D([x, x + 0.022], [0.06, 0.06], color=C_NS, lw=0.6, transform=fig.transFigure))
        fig.text(x + 0.026, 0.06, "< 3/5", va="center", fontsize=6)
        fig.text(0.02, 0.015, "Analyses supporting the change (Pp > .95, same sign)", fontsize=5.5, color="0.4")

        # ---------------- C: estimate matrix for supported connections
        keep = sup[sup["support"] >= MIN_SUPPORT].copy()
        keep["order"] = keep["effect"].map({"local": 0, "global": 1})
        keep = keep.sort_values(["order", "support"], ascending=[True, False]).reset_index(drop=True)
        axC = fig.add_axes([0.60, 0.16, 0.20, 0.74])
        E = keep[[f"Ep_{a}" for a in ANALYSES]].to_numpy(float)
        lim = np.ceil(np.nanmax(np.abs(E)) * 10) / 10
        im = axC.imshow(E, cmap="RdBu_r", norm=TwoSlopeNorm(0, -lim, lim), aspect="auto")
        for i in range(len(keep)):
            for j, a in enumerate(ANALYSES):
                if keep.loc[i, f"cred_{a}"]:
                    axC.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="k", lw=0.7))
            axC.text(len(ANALYSES) - 0.35, i, f"{keep.loc[i, 'support']}/5", va="center", fontsize=5.5)
        axC.set_yticks(range(len(keep)), [pretty(c) for c in keep["connection"]], fontsize=5.5)
        for tl, eff in zip(axC.get_yticklabels(), keep["effect"]):
            tl.set_color(C_EFF[eff])
        axC.set_xticks(range(len(ANALYSES)), [ALAB[a] for a in ANALYSES], rotation=90, fontsize=5.5)
        axC.tick_params(length=0)
        nloc = int((keep["effect"] == "local").sum())
        axC.axhline(nloc - 0.5, color="k", lw=0.8)
        for sp in axC.spines.values():
            sp.set_visible(False)
        cax = fig.add_axes([0.60, 0.935, 0.11, 0.02])
        cb = fig.colorbar(im, cax=cax, orientation="horizontal")
        cb.ax.tick_params(labelsize=5, length=1.5, pad=1)
        cb.set_label("log gain", fontsize=5.5, labelpad=1)
        cb.ax.xaxis.set_label_position("top")
        fig.text(0.49, 0.985, "C", fontsize=9, weight="bold", va="top")

        # ---------------- D: fit quality
        axD = fig.add_axes([0.915, 0.30, 0.08, 0.60])
        for j, a in enumerate(ANALYSES):
            v = r2[a]
            axD.scatter(np.full(len(v), j) + np.random.default_rng(j).uniform(-0.15, 0.15, len(v)), v, s=3,
                        color="0.55", lw=0)
            axD.plot([j - 0.3, j + 0.3], [np.median(v)] * 2, color="k", lw=1)
        axD.set_xticks(range(len(ANALYSES)), [ALAB[a].replace("\n", " ") for a in ANALYSES], rotation=90, fontsize=5)
        axD.set_ylim(0.7, 1.0)
        axD.set_ylabel("Variance explained")
        sns.despine(ax=axD)
        fig.text(0.85, 0.985, "D", fontsize=9, weight="bold", va="top")

        FIG_DIR.mkdir(exist_ok=True)
        fig.savefig(FIG_DIR / "Figure_DCM_robust.pdf")
        fig.savefig(FIG_DIR / "Figure_DCM_robust.svg")
        svg = FIG_DIR / "Figure_DCM_robust.svg"
        svg.write_text(svg.read_text().replace("Liberation Sans", "Arial"))
        fig.savefig(FIG_DIR / "Figure_DCM_robust.png", dpi=400)
        print(keep[["effect", "connection", "sign", "support"]].to_string(index=False))
        print(json.dumps({a: dict(median=float(np.median(v)), min=float(v.min()), n=len(v)) for a, v in r2.items()}, indent=1))

    return main()


run_figure('DCM robustness', make_figure_dcm_robust)

plt.close('all')
for f in ['Figure_DCM_robust.png']:
    p = PROJECT_ROOT / 'figures' / f
    display(Image(filename=str(p), width=900) if p.exists() else f'missing: {p}')

### Supplementary — main DCM (0/1 coding, 0–600 ms)

`scripts/22_figure_dcm.py`. From the script's header:

```text
22_figure_dcm.py — DCM figure (173.695 x 68 mm): how local and global deviance change the network.

  A  Local deviance: group (PEB, Bayesian model average) modulation of each forward / backward
     connection; coloured = posterior probability > .95 (orange = stronger, blue = weaker than for
     standards), width ∝ |effect| (log scale: +0.5 ≈ 1.6x); grey = not credible. Dashed = fixed
     lateral IFG–IPL connections; arrows below A1 = auditory input.
  B  Global deviance: same.
  C  Family comparison (which connection types each effect modulates): posterior probability.
  D  Left IPL -> STG backward connection per participant: modulation by local vs global deviance
     (individual DCM estimates; lines join a participant), the connection whose modulation reverses.
  Fit quality (variance explained per participant) is reported in derivatives/dcm/figure_dcm_stats.json.

Inputs : derivatives/dcm/peb/{BMR.mat, family_local.mat, family_global.mat}, derivatives/dcm/fits/DCM_*_full.mat
Outputs: figures/Figure_DCM.pdf / .svg / .png; derivatives/dcm/figure_dcm_stats.json

Usage:  python scripts/22_figure_dcm.py
```

In [ ]:
# ---- 22_figure_dcm.py (inline; rebuild the notebook after editing the script)
def make_figure_dcm_main():
    from pathlib import Path
    import json
    import re

    import numpy as np
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
    import seaborn as sns
    from scipy.io import loadmat
    from scipy import stats

    # ----------------------------------------------------------------------------- CONFIG
    ROOT = PROJECT_ROOT
    DCM_DIR = ROOT / "derivatives" / "dcm"
    FIG_DIR = ROOT / "figures"
    SIZE_MM = (173.695, 68.0)
    SRC = ["lA1", "rA1", "lSTG", "rSTG", "lIFG", "rIFG", "lIPL", "rIPL"]          # order used in dcm_02_fit.m
    LABEL = {"lA1": "A1", "rA1": "A1", "lSTG": "STG", "rSTG": "STG", "lIFG": "IFG", "rIFG": "IFG",
             "lIPL": "IPL", "rIPL": "IPL"}
    POS = {"lA1": (-1.0, 0.0), "lSTG": (-1.0, 1.0), "lIFG": (-1.55, 2.05), "lIPL": (-0.45, 2.05),
           "rA1": (1.0, 0.0), "rSTG": (1.0, 1.0), "rIFG": (1.55, 2.05), "rIPL": (0.45, 2.05)}
    FWD = [(3, 1), (4, 2), (5, 3), (6, 4), (7, 3), (8, 4)]                        # (to, from), 1-based
    LAT = [(5, 7), (6, 8)]
    PP_THR = 0.95
    C_UP, C_DOWN, C_NS = "#D55E00", "#0072B2", "0.82"
    C_EFF = {"local": "#C44E52", "global": "#8172B3"}
    FAM_NAMES = ["none", "fwd", "bwd", "fwd+bwd"]
    KEY = (3, 7)                                                                  # lIPL -> lSTG (to, from)
    TEXT_RC = {"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
               "font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "Helvetica", "DejaVu Sans"],
               "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 6.5, "xtick.labelsize": 6, "ytick.labelsize": 6,
               "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
               "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5,
               "axes.labelpad": 2}
    # ------------------------------------------------------------------------------------


    def dense(x):
        return x.toarray() if hasattr(x, "toarray") else np.asarray(x)


    def load_bma():
        """Group BMA: dict (effect, to, from) -> (Ep, Pp); Pp = P(|effect| > 0) under the Gaussian posterior."""
        B = loadmat(DCM_DIR / "peb" / "BMR.mat", squeeze_me=True, struct_as_record=False)["BMA"]
        Ep, var = dense(B.Ep).ravel(), dense(B.Cp).ravel()
        var = np.diag(dense(B.Cp)) if dense(B.Cp).ndim == 2 and dense(B.Cp).shape[0] == dense(B.Cp).shape[1] > 1 else var
        sd = np.sqrt(np.clip(var, 0, None))
        out = {}
        for i, name in enumerate(np.atleast_1d(B.Pnames)):
            k, r, c = map(int, re.findall(r"\d+", str(name)))
            pp = stats.norm.cdf(abs(Ep[i]) / sd[i]) if sd[i] > 0 else 0.5
            out[("local" if k == 1 else "global", r, c)] = (float(Ep[i]), float(pp))
        return out


    def load_subjects():
        """Per-participant B estimates and variance explained from the individual full DCMs."""
        B, r2 = {}, {}
        for f in sorted((DCM_DIR / "fits").glob("DCM_sub-*_full.mat")):
            sub = f.stem.split("_")[1]
            D = loadmat(f, squeeze_me=True, struct_as_record=False)["DCM"]
            Bk = [np.asarray(b) for b in np.atleast_1d(D.Ep.B)]
            B[sub] = {"local": Bk[0], "global": Bk[1]}
            H = [np.asarray(h) for h in np.atleast_1d(D.H)]
            R = [np.asarray(x) for x in np.atleast_1d(D.R)]
            r2[sub] = float(1 - sum((x ** 2).sum() for x in R) / sum(((h + x) ** 2).sum() for h, x in zip(H, R)))
        return B, r2


    def draw_network(ax, bma, eff):
        ax.set_xlim(-2.05, 2.05)
        ax.set_ylim(-0.55, 2.45)
        ax.set_aspect("equal")
        ax.set_axis_off()
        for a, b in LAT:                                                          # fixed lateral connections
            (x0, y0), (x1, y1) = POS[SRC[a - 1]], POS[SRC[b - 1]]
            ax.plot([x0, x1], [y0, y1], ls=(0, (2, 2)), color="0.7", lw=0.6, zorder=1)
        for side in ("l", "r"):                                                   # auditory input
            x, y = POS[f"{side}A1"]
            ax.add_patch(FancyArrowPatch((x, y - 0.5), (x, y - 0.2), arrowstyle="-|>", mutation_scale=6,
                                         color="0.45", lw=0.7))
        for to, frm in FWD:
            for direction, (r, c) in (("fwd", (to, frm)), ("bwd", (frm, to))):
                ep, pp = bma[(eff, r, c)]
                sig = pp > PP_THR
                col = (C_UP if ep > 0 else C_DOWN) if sig else C_NS
                lw = 0.5 + 3.2 * abs(ep) if sig else 0.5
                p0, p1 = POS[SRC[c - 1]], POS[SRC[r - 1]]
                rad = 0.28 if direction == "fwd" else 0.28
                ax.add_patch(FancyArrowPatch(p0, p1, connectionstyle=f"arc3,rad={rad}", arrowstyle="-|>",
                                             mutation_scale=5 + 2 * lw, lw=lw, color=col, shrinkA=8, shrinkB=8,
                                             zorder=3 if sig else 2))
        for name, (x, y) in POS.items():
            ax.add_patch(FancyBboxPatch((x - 0.22, y - 0.13), 0.44, 0.26, boxstyle="round,pad=0,rounding_size=0.08",
                                        fc="white", ec="0.3", lw=0.7, zorder=4))
            ax.text(x, y, LABEL[name], ha="center", va="center", fontsize=6.5, zorder=5)
        ax.text(-1.0, 2.42, "Left", ha="center", va="top", fontsize=6, color="0.4")
        ax.text(1.0, 2.42, "Right", ha="center", va="top", fontsize=6, color="0.4")


    def main():
        bma = load_bma()
        fam = {e: np.atleast_1d(loadmat(DCM_DIR / "peb" / f"family_{e}.mat", squeeze_me=True,
                                        struct_as_record=False)[f"fam{e[0].upper()}"].family.post).astype(float)
               for e in ("local", "global")}
        Bs, r2 = load_subjects()
        subs = sorted(Bs)
        n = len(subs)

        sns.set_theme(style="ticks", rc=TEXT_RC)
        mm = 1 / 25.4
        fig = plt.figure(figsize=(SIZE_MM[0] * mm, SIZE_MM[1] * mm))
        axA = fig.add_axes([0.0, 0.1, 0.30, 0.84])
        axB = fig.add_axes([0.31, 0.1, 0.30, 0.84])
        for ax, eff, lab in ((axA, "local", "A"), (axB, "global", "B")):
            draw_network(ax, bma, eff)
            ax.set_title(f"{eff.capitalize()} deviance", color=C_EFF[eff], weight="bold", pad=2)
            fig.text(ax.get_position().x0 + 0.005, 0.985, lab, fontsize=9, weight="bold", va="top")
        # legend for the networks
        lx = 0.05
        for col, txt in ((C_UP, "stronger"), (C_DOWN, "weaker"), (C_NS, "n.s.")):
            fig.add_artist(plt.Line2D([lx, lx + 0.025], [0.045, 0.045], color=col, lw=1.6, transform=fig.transFigure))
            fig.text(lx + 0.03, 0.045, txt, va="center", fontsize=6)
            lx += 0.1
        fig.text(lx + 0.01, 0.045, f"(Pp > {PP_THR:.2f}; line width scales with effect size)", va="center", fontsize=6, color="0.4")

        # ---------------- C: family posteriors
        axC = fig.add_axes([0.69, 0.62, 0.29, 0.28])
        w = 0.38
        x = np.arange(4)
        for k, eff in enumerate(("local", "global")):
            axC.bar(x + (k - 0.5) * w, fam[eff], width=w, color=C_EFF[eff], label=eff)
        axC.set_xticks(x, FAM_NAMES)
        axC.set_ylim(0, 1.05)
        axC.set_yticks([0, 0.5, 1])
        axC.set_ylabel("Family posterior")
        axC.legend(frameon=False, loc="upper left", handlelength=1, borderaxespad=0.1, fontsize=6)
        sns.despine(ax=axC)
        fig.text(0.625, 0.985, "C", fontsize=9, weight="bold", va="top")
        axC.set_title("Connection types modulated", pad=3)

        # ---------------- D: key connection per participant
        axD = fig.add_axes([0.69, 0.13, 0.29, 0.33])
        to, frm = KEY
        loc = np.array([Bs[s]["local"][to - 1, frm - 1] for s in subs])
        glo = np.array([Bs[s]["global"][to - 1, frm - 1] for s in subs])
        for a, b in zip(loc, glo):
            axD.plot([0, 1], [a, b], color="0.75", lw=0.5, zorder=1)
        axD.scatter(np.zeros(n), loc, s=8, color=C_EFF["local"], zorder=2, lw=0)
        axD.scatter(np.ones(n), glo, s=8, color=C_EFF["global"], zorder=2, lw=0)
        axD.axhline(0, color="k", lw=0.4)
        tt = stats.ttest_rel(loc, glo)
        axD.set_xticks([0, 1], ["local", "global"])
        axD.set_xlim(-0.4, 1.4)
        axD.set_ylabel("Modulation (log gain)")
        axD.set_title(f"Left IPL → STG (backward)", pad=3)
        ptxt = "p < .001" if tt.pvalue < 0.001 else f"p = {tt.pvalue:.3f}".replace("0.", ".")
        axD.text(0.5, 1.0, f"t({n - 1}) = {tt.statistic:.1f}, {ptxt}\n{int((loc > glo).sum())}/{n} local > global",
                 transform=axD.transAxes, ha="center", va="top", fontsize=5.5)
        sns.despine(ax=axD)
        fig.text(0.625, 0.52, "D", fontsize=9, weight="bold", va="top")

        out = dict(r2_per_subject=r2, r2_median=float(np.median(list(r2.values()))),
                   r2_range=[float(min(r2.values())), float(max(r2.values()))],
                   family_posterior={e: dict(zip(FAM_NAMES, map(float, fam[e]))) for e in fam},
                   bma={f"{e}|{SRC[c - 1]}->{SRC[r - 1]}": dict(Ep=v[0], Pp=v[1]) for (e, r, c), v in bma.items()},
                   key_connection=dict(name="lIPL->lSTG", local_mean=float(loc.mean()), global_mean=float(glo.mean()),
                                       n_local_pos=int((loc > 0).sum()), n_global_neg=int((glo < 0).sum()),
                                       paired_t=float(tt.statistic), p=float(tt.pvalue)))
        FIG_DIR.mkdir(exist_ok=True)
        fig.savefig(FIG_DIR / "Figure_DCM.pdf")
        fig.savefig(FIG_DIR / "Figure_DCM.svg")
        svg = FIG_DIR / "Figure_DCM.svg"                 # SVG names fonts: Liberation Sans -> Arial (same metrics)
        svg.write_text(svg.read_text().replace("Liberation Sans", "Arial"))
        fig.savefig(FIG_DIR / "Figure_DCM.png", dpi=400)
        (DCM_DIR / "figure_dcm_stats.json").write_text(json.dumps(out, indent=2))
        print(json.dumps({k: v for k, v in out.items() if k != "bma"}, indent=2))

    return main()


run_figure('DCM main model', make_figure_dcm_main)

plt.close('all')
for f in ['Figure_DCM.png']:
    p = PROJECT_ROOT / 'figures' / f
    display(Image(filename=str(p), width=900) if p.exists() else f'missing: {p}')

### Supplementary — ERPs predicted by the DCM vs observed

`scripts/23_dcm_predicted_erps.py`. From the script's header:

```text
23_dcm_predicted_erps.py — the ERPs generated by the fitted DCMs vs the observed ERPs (173.695 x 100 mm).

Rows: frontocentral ROI (local effect) and parietal ROI (global effect), as in Figures 1–2.
  A/D  observed (solid) and model-predicted (dashed) ERPs: standard vs deviant
  B/E  observed vs predicted difference wave (LD − LS; GD − GS), with the correlation over time
  C/F  the difference wave at each model source (regional pyramidal-cell output, the signal the
       model projects to the scalp), showing which sources generate the MMN and the P3b in the model

How the waves are obtained from each participant's full DCM (dcm_02_fit.m):
  predicted channel data = H{trial} · U'   (H: model prediction in the 8 spatial modes; U: modes)
  observed  channel data = (H + R){trial} · U'  (data as the model saw them: same modes, same window)
  source output = Σ_states K{trial} · J. Amplitudes are in DCM's internal units (a.u.): SPM rescales
  the data before fitting, so only shapes and relative sizes are meaningful. The "observed" waves are
  the data as the DCM sees them — projected onto 8 modes and with the window mean removed (options.h
  = 1), which shifts slow components such as the P3b; compare shapes with Figure 1 accordingly.
LS/LD/GS/GD are averages of the two 2x2 cells that share that level (equal weights).
Window 0–600 ms (the DCM window). Group mean ± SE over participants.

Inputs : derivatives/dcm/fits/DCM_sub-*_full.mat
Outputs: figures/DCM_predicted_ERPs.pdf / .svg / .png; derivatives/dcm/predicted_erp_fit.json

Usage:  python scripts/23_dcm_predicted_erps.py
```

In [ ]:
# ---- 23_dcm_predicted_erps.py (inline; rebuild the notebook after editing the script)
def make_figure_dcm_predicted():
    from pathlib import Path
    import json

    import numpy as np
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy.io import loadmat

    # ----------------------------------------------------------------------------- CONFIG
    ROOT = PROJECT_ROOT
    FIT_DIR = ROOT / "derivatives" / "dcm" / "fits"
    FIG_DIR = ROOT / "figures"
    SIZE_MM = (173.695, 100.0)
    CONDS = ["LSGS", "LDGS", "LSGD", "LDGD"]                       # trial order in the DCM
    LEVELS = {"LS": [0, 2], "LD": [1, 3], "GS": [0, 1], "GD": [2, 3]}
    ROWS = [("local", ["Fz", "FC1", "FC2", "F1", "F2", "Cz"], ("LS", "LD"), "Frontocentral ROI", (80, 140)),
            ("global", ["Pz", "P1", "P2", "CPz", "POz", "CP1", "CP2"], ("GS", "GD"), "Parietal ROI", (300, 600))]
    SRC = ["lA1", "rA1", "lSTG", "rSTG", "lIFG", "rIFG", "lIPL", "rIPL"]
    SRC_COL = {"A1": "#9DB4C0", "STG": "#4C72B0", "IFG": "#DD8452", "IPL": "#55A868"}
    C_STD = "0.35"
    C_DEV = {"local": "#C44E52", "global": "#8172B3"}
    TEXT_RC = {"pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
               "font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "Helvetica", "DejaVu Sans"],
               "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 6.5, "xtick.labelsize": 6, "ytick.labelsize": 6,
               "legend.fontsize": 5.8, "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
               "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5,
               "axes.labelpad": 2}
    # ------------------------------------------------------------------------------------


    def dense(x):
        return x.toarray() if hasattr(x, "toarray") else np.asarray(x, float)


    def load_subject(f):
        """Observed and predicted channel waves (cond x chan x time, µV), source output (cond x src x time)."""
        D = loadmat(f, squeeze_me=True, struct_as_record=False)["DCM"]
        U = dense(D.M.U)                                           # chan x modes
        scale = float(D.xY.scale)
        H = [dense(h) for h in np.atleast_1d(D.H)]
        R = [dense(r) for r in np.atleast_1d(D.R)]
        pred = np.stack([(h @ U.T).T for h in H]) / scale
        obs = np.stack([((h + r) @ U.T).T for h, r in zip(H, R)]) / scale
        J = dense(D.Eg.J).ravel()
        js = np.flatnonzero(J)                                     # states that contribute to the EEG
        ns = len(SRC)
        src = []
        for k in np.atleast_1d(D.K):
            k = dense(k)                                           # time x (states x sources), state-major
            src.append(sum(J[s] * k[:, i * ns:(i + 1) * ns] for i, s in enumerate(js)).T)
        return dict(obs=obs, pred=pred, src=np.stack(src), chans=list(D.xY.name), t=np.asarray(D.xY.pst, float))


    def level(X, lv):
        return X[:, LEVELS[lv]].mean(1)


    def main():
        subs = [load_subject(f) for f in sorted(FIT_DIR.glob("DCM_sub-*_full.mat"))]
        n = len(subs)
        t = subs[0]["t"]
        chans = subs[0]["chans"]
        obs = np.stack([s["obs"] for s in subs])                   # subj x cond x chan x time
        pred = np.stack([s["pred"] for s in subs])
        src = np.stack([s["src"] for s in subs])                   # subj x cond x src x time

        sns.set_theme(style="ticks", rc=TEXT_RC)
        mm = 1 / 25.4
        fig, axes = plt.subplots(2, 3, figsize=(SIZE_MM[0] * mm, SIZE_MM[1] * mm),
                                 gridspec_kw=dict(left=0.075, right=0.985, bottom=0.1, top=0.92, wspace=0.38,
                                                  hspace=0.5, width_ratios=[1, 1, 1.1]))
        out = {}

        def band(ax, X, col, ls="-", lw=1.0, lab=None, fill=True):
            m, se = X.mean(0), X.std(0, ddof=1) / np.sqrt(len(X))
            ax.plot(t, m, color=col, ls=ls, lw=lw, label=lab)
            if fill:
                ax.fill_between(t, m - se, m + se, color=col, alpha=0.15, lw=0)
            return m

        for r, (eff, roi, (std, dev), roiname, win) in enumerate(ROWS):
            ci = [chans.index(c) for c in roi]
            O = obs[:, :, ci].mean(2)                              # subj x cond x time
            P = pred[:, :, ci].mean(2)
            # ---- conditions
            ax = axes[r, 0]
            ax.axvspan(*win, color="0.93", lw=0, zorder=0)
            band(ax, level(O, std), C_STD, lab=f"{std} observed")
            band(ax, level(O, dev), C_DEV[eff], lab=f"{dev} observed")
            band(ax, level(P, std), C_STD, ls="--", lw=0.9, lab=f"{std} model", fill=False)
            band(ax, level(P, dev), C_DEV[eff], ls="--", lw=0.9, lab=f"{dev} model", fill=False)
            ax.axhline(0, color="k", lw=0.4)
            ax.set_ylabel("Amplitude (a.u.)")
            ax.set_title(f"{roiname}: {std} vs {dev}", pad=3)
            ax.legend(frameon=False, ncol=2, loc="upper left" if r == 1 else "lower right", handlelength=1.6,
                      columnspacing=0.6, borderaxespad=0.1)
            # ---- difference
            ax = axes[r, 1]
            ax.axvspan(*win, color="0.93", lw=0, zorder=0)
            dO = level(O, dev) - level(O, std)
            dP = level(P, dev) - level(P, std)
            mo = band(ax, dO, "k", lab="observed")
            mp = band(ax, dP, C_DEV[eff], ls="--", lw=1.0, lab="model", fill=False)
            rr = float(np.corrcoef(mo, mp)[0, 1])
            per_sub = [float(np.corrcoef(a, b)[0, 1]) for a, b in zip(dO, dP)]
            ax.axhline(0, color="k", lw=0.4)
            ax.set_ylabel(f"{dev} − {std} (a.u.)")
            ax.set_title(f"Difference wave (group r = {rr:.2f})", pad=3)
            ax.legend(frameon=False, loc="upper left" if r == 1 else "lower right", handlelength=1.6, borderaxespad=0.1)
            # ---- sources
            ax = axes[r, 2]
            ax.axvspan(*win, color="0.93", lw=0, zorder=0)
            dS = level(src, dev) - level(src, std)                  # subj x src x time
            peak = np.abs(dS.mean(0)).max()
            for k, s in enumerate(SRC):
                reg = s[1:]
                ax.plot(t, dS[:, k].mean(0) / peak, color=SRC_COL[reg], lw=1.0, ls="-" if s[0] == "l" else ":",
                        label=f"{'L' if s[0] == 'l' else 'R'} {reg}")
            ax.axhline(0, color="k", lw=0.4)
            ax.set_ylabel("Source output (norm.)")
            ax.set_title(f"{dev} − {std} at each model source", pad=3)
            if r == 1:
                ax.legend(frameon=False, ncol=2, loc="upper right", handlelength=1.6, columnspacing=0.6,
                          borderaxespad=0.1)
            win_idx = (t >= win[0]) & (t <= win[1])
            out[eff] = dict(group_r_difference_wave=rr, per_subject_r_median=float(np.median(per_sub)),
                            per_subject_r_range=[float(min(per_sub)), float(max(per_sub))],
                            source_share_in_window={s: float(np.abs(dS[:, k].mean(0)[win_idx]).mean()
                                                              / np.abs(dS.mean(0)[:, win_idx]).mean(1).sum())
                                                    for k, s in enumerate(SRC)})
            for ax in axes[r]:
                ax.axvline(0, color="k", lw=0.5)
                ax.set_xlim(0, 600)
                ax.set_xticks([0, 200, 400, 600])
                sns.despine(ax=ax)
                if r == 1:
                    ax.set_xlabel("Time from 5th sound (ms)")
        for k, ax in enumerate(axes.ravel()):
            ax.text(-0.27, 1.12, "ABCDEF"[k], transform=ax.transAxes, fontsize=9, weight="bold", va="top")

        FIG_DIR.mkdir(exist_ok=True)
        fig.savefig(FIG_DIR / "DCM_predicted_ERPs.pdf")
        fig.savefig(FIG_DIR / "DCM_predicted_ERPs.svg")
        svg = FIG_DIR / "DCM_predicted_ERPs.svg"
        svg.write_text(svg.read_text().replace("Liberation Sans", "Arial"))
        fig.savefig(FIG_DIR / "DCM_predicted_ERPs.png", dpi=400)
        (ROOT / "derivatives" / "dcm" / "predicted_erp_fit.json").write_text(json.dumps(out, indent=2))
        print(json.dumps(out, indent=2))

    return main()


run_figure('DCM predicted ERPs', make_figure_dcm_predicted)

plt.close('all')
for f in ['DCM_predicted_ERPs.png']:
    p = PROJECT_ROOT / 'figures' / f
    display(Image(filename=str(p), width=900) if p.exists() else f'missing: {p}')

## 5. Validation of the methods on simulated data

Each check simulates data where the answer is known and confirms the method returns it.

| Check | Method | Expected |
|---|---|---|
| V1 | leaky Beta-Bernoulli learner | closed-form posterior; reset and forgetting behave as specified |
| V2 | random-effects BMS / PXP / BOR | finds the true majority model; BOR ≈ 1 and PXP ≈ 1/K under the null |
| V3 | nested CV | selects the generating *model family*; shows how well half-life / reset are identifiable at the real SNR |
| V4 | temporal cluster test | false-positive rate ≈ 5% on null data with ERP-like autocorrelation |
| V5 | 2 × 2 rm-ANOVA | each F equals the squared t of its contrast |
| V6 | added-variable slope | ≈ 0 for a regressor with no unique effect, despite a raw correlation |
| V7 | eLORETA localisation | a simulated left auditory-cortex source is localised there, and how far it leaks |

In [ ]:
# V1: learner against the closed form. No forgetting: p(change) before trial t = (k + 1) / (t + 2).
rng = np.random.default_rng(1)
y = rng.integers(0, 2, 200)
S = leaky_surprise(y, np.inf, None)
k = np.concatenate([[0], np.cumsum(y)[:-1]]); tt_ = np.arange(200)
mu = (k + 1) / (tt_ + 2)
assert np.allclose(S, -np.log2(np.where(y == 1, mu, 1 - mu))), "learner != closed form"
# reset: surprise after a reset equals surprise of a fresh learner on the same outcomes
r_ = np.zeros(200, bool); r_[100] = True
assert np.allclose(leaky_surprise(y, np.inf, r_)[100:], leaky_surprise(y[100:], np.inf, None))
# forgetting: half-life h halves the weight of an observation after h further observations
h = 10; lam = 2 ** (-1 / h); assert np.isclose(lam ** h, 0.5)
# block structure: the first deviant of a 'repeat' block is maximally surprising after a reset
print("V1 OK: learner matches the closed-form posterior; reset and forgetting behave as specified")

In [ ]:
# V2: BMS. Simulate 15 subjects x 4 models; each subject's true model drawn from frequencies r,
# log evidence = true model gets +d, others 0, plus noise.
def sim_L(freqs, d=4.0, sd=1.0, n_sub=15, seed=0):
    g = np.random.default_rng(seed)
    true = g.choice(len(freqs), n_sub, p=freqs)
    L = g.normal(0, sd, (n_sub, len(freqs))); L[np.arange(n_sub), true] += d
    return L
wins, bors = [], []
for s in range(200):
    b = rfx_bms(sim_L([0.6, 0.2, 0.1, 0.1], seed=s), n_samples=20000, seed=s)
    wins.append(np.argmax(b["pxp"]) == 0)
null = [rfx_bms(np.random.default_rng(s).normal(0, 1, (15, 4)), n_samples=20000, seed=s) for s in range(200)]
print(f"V2: majority model (60%) has the highest PXP in {100*np.mean(wins):.0f}% of 200 simulations")
print(f"    null (no differences): mean BOR = {np.mean([b['bor'] for b in null]):.2f}, mean max PXP = {np.mean([b['pxp'].max() for b in null]):.2f} (1/K = 0.25), "
      f"max PXP > 0.9 in {100*np.mean([b['pxp'].max() > 0.9 for b in null]):.1f}%")

In [ ]:
# V3: nested CV recovery on a real sequence (sub-02). Generate single-trial data from M2 with a known
# half-life and reset (truth = 50-quintlet half-life, reset at block start), at the empirical noise level
# (SD of that participant's single-trial amplitudes), with an effect size x1, x3 and x10 the empirical
# one (-0.6 µV per SD of surprise), and ask which configuration the inner CV selects.
s0 = "sub-02"; tr0, bank0 = load_trials(s0), regressor_bank(s0)
truth = ("loc", 50, True)
X = design_matrix(tr0, bank0, [truth])
noise_sd = tr0["local_80-140ms"].std()
# (a) how different are the candidate regressors at all? correlation of the truth with every local config
loc_keys = [k for k in bank0 if k[0] == "loc"]
R = np.corrcoef(np.array([bank0[k][tr0["seq_idx"].to_numpy()] for k in loc_keys]))
print("V3a: correlation of the generating regressor with each local configuration (half-life, reset):")
print("     " + ", ".join(f"{k[1]}/{'R' if k[2] else '-'}: {R[loc_keys.index(truth), j]:.3f}" for j, k in enumerate(loc_keys)))
# (b) recovery as a function of SNR (10 simulations per level; the code is correct if recovery -> 100%)
for scale in [1, 3, 10]:
    sel = []
    for rep in range(10):
        y = X @ np.array([0, 0, -0.6 * scale]) + np.random.default_rng(rep).normal(0, noise_sd, len(tr0))
        _, chosen = nested_cv(tr0, y, bank0, "M2 local")
        sel.append(pd.Series([c[0] for c in chosen]).mode()[0])
    print(f"V3b: effect x{scale:<2}: CV's modal choice = truth in {sum(c == truth for c in sel)}/10; "
          f"choices: {pd.Series(sel).value_counts().to_dict()}")
# (c) model identity at the empirical SNR: M2 should still beat M0, M1 and M3 on held-out data
y = X @ np.array([0, 0, -0.6]) + np.random.default_rng(99).normal(0, noise_sd, len(tr0))
print("V3c: held-out log-lik at empirical SNR:", {m.split()[0]: round(nested_cv(tr0, y, bank0, m)[0], 1) for m in MODELS[:4]})

In [ ]:
# V4: false-positive rate of the temporal cluster test on null ERP-like data (15 subjects, 1/f-ish noise
# smoothed to ~30 Hz, sampled like the real data 0-700 ms).
from scipy.ndimage import gaussian_filter1d
import warnings; warnings.filterwarnings("ignore", message="No clusters found")
tt0 = times[(times >= 0) & (times <= 0.7)]
fp = 0; n_sim = 200
for s in range(n_sim):
    g = np.random.default_rng(1000 + s)
    Xn = gaussian_filter1d(g.normal(0, 1, (15, len(tt0))), sigma=5, axis=1)
    _, cl, pv, _ = mne.stats.permutation_cluster_1samp_test(Xn, threshold=stats.t.ppf(0.975, 14), n_permutations=500,
                                                           tail=0, seed=s, out_type="indices", verbose=False)
    fp += np.any(pv < 0.05)
print(f"V4: family-wise false-positive rate = {fp/n_sim:.3f} (nominal 0.05; 95% MC interval ~0.02-0.08)")

In [ ]:
# V5: 2x2 rm-ANOVA vs contrast t-tests (real cell means): F(1,14) = t^2 for each effect
for comp in ["local", "global"]:
    aov, piv = anova[comp]; aov = aov.set_index("Source")
    con = {"local": (piv.LDGS + piv.LDGD) - (piv.LSGS + piv.LSGD),
           "global": (piv.LSGD + piv.LDGD) - (piv.LSGS + piv.LDGS),
           "local * global": (piv.LDGD - piv.LSGD) - (piv.LDGS - piv.LSGS)}
    for eff, c in con.items():
        t2_ = stats.ttest_1samp(c, 0).statistic ** 2
        assert np.isclose(t2_, aov.loc[eff, "F"], rtol=1e-6), (comp, eff)
print("V5 OK: every ANOVA F equals the squared contrast t")

# V6: added-variable slope. x1 drives y; x2 correlates with x1 (r ~ .2, as S_loc vs S_glob) but has no effect.
g = np.random.default_rng(5); raw, part = [], []
for s in range(15):
    x1 = g.normal(size=900); x2 = 0.2 * x1 + g.normal(size=900); y = -0.5 * x1 + g.normal(size=900) * 3
    raw.append(np.polyfit(x2, y, 1)[0]); part.append(np.polyfit(resid(x2, x1), resid(y, x1), 1)[0])
print(f"V6: raw slope of the inert regressor t = {stats.ttest_1samp(raw,0).statistic:.1f} (spurious), "
      f"added-variable slope t = {stats.ttest_1samp(part,0).statistic:.2f}, p = {stats.ttest_1samp(part,0).pvalue:.2f} (correctly null)")

In [ ]:
# V7: eLORETA localisation of a simulated source in left auditory cortex (template head, one covariance)
if have_template and need_stcs:
    fwd_fixed = mne.convert_forward_solution(fwd, surf_ori=True, force_fixed=True, use_cps=True)
    idx = np.where(np.isin(vert[0], ac["lh"].vertices))[0]
    x = np.zeros(sum(len(v) for v in vert)); x[idx] = 1e-9 * mne.label_sign_flip(ac["lh"], src)
    ev = mne.EvokedArray(np.tile((fwd_fixed["sol"]["data"] @ x)[:, None], (1, 3)), inf, tmin=0)
    ev.set_eeg_reference("average", projection=True)
    inv = mne.minimum_norm.make_inverse_operator(inf, fwd, mne.read_cov(SRC_DIR / f"{subs[0]}-cov.fif"), loose=LOOSE, depth=DEPTH)
    e = mne.minimum_norm.apply_inverse(ev, inv, LAMBDA2, method="eLORETA", pick_ori="normal", method_params=dict(max_iter=60))
    amp = np.abs(e.data[:, 0]); rows = []
    for lb in labs:
        if "unknown" in lb.name: continue
        h = 0 if lb.hemi == "lh" else 1
        ii = np.where(np.isin(vert[h], lb.vertices))[0] + (0 if h == 0 else len(vert[0]))
        if len(ii): rows.append(dict(label=lb.name, mean_amp=amp[ii].mean()))
    v7 = pd.DataFrame(rows).sort_values("mean_amp", ascending=False)
    v7["rel"] = v7.mean_amp / v7.mean_amp.max()
    peak = int(np.argmax(amp)); hp = 0 if peak < len(vert[0]) else 1
    vpk = vert[hp][peak - (0 if hp == 0 else len(vert[0]))]
    peak_lab = next((lb.name for lb in labs if lb.hemi == ["lh", "rh"][hp] and vpk in lb.vertices), "?")
    in_ac = amp[idx].sum() / amp.sum()
    print(f"V7: peak of the estimate in {peak_lab}; {100*in_ac:.0f}% of total |current| inside the simulated AC patch")
    print("    mean |current| by region (top 8, relative to the strongest region):")
    print(v7.head(8)[["label", "rel"]].round(2).to_string(index=False))

## 6. Every Results number vs this notebook

`draft` = the value in the Results draft (rounded as reported); `notebook` = recomputed above.
`match` = equal at the reported precision.

In [ ]:
DRAFT = {   # (section, quantity) -> (value in the Results draft, decimals as reported)
    (1, "MMN mean (µV)"): (-1.22, 2), (1, "MMN t"): (-9.4, 1), (1, "MMN 95% CI half-width"): (0.28, 2), (1, "MMN peak (ms)"): (106, 0),
    (1, "P3b mean (µV)"): (2.35, 2), (1, "P3b t"): (5.4, 1), (1, "P3b 95% CI half-width"): (0.94, 2), (1, "P3b peak (ms)"): (432, 0),
    (1, "ANOVA local-window local F"): (79.2, 1), (1, "ANOVA local-window global F"): (31.5, 1), (1, "ANOVA local-window local * global F"): (12.9, 1),
    (1, "ANOVA global-window local F"): (0.8, 1), (1, "ANOVA global-window global F"): (28.8, 1), (1, "ANOVA global-window local * global F"): (3.6, 1),
    (1, "local simple effect LD: GD vs GS t"): (-5.0, 1), (1, "local simple effect LS: GD vs GS t"): (-2.5, 1),
    (1, "global simple effect LS: GD vs GS t"): (4.9, 1), (1, "global simple effect LD: GD vs GS t"): (5.7, 1),
    (1, "MMN Fz linked mastoids (µV)"): (-3.17, 2), (1, "Mastoid inversion (µV)"): (1.77, 2),
    (2, "PXP M2 local (early)"): (0.94, 2), (2, "BOR early"): (0.022, 3), (2, "PXP M3 global (late)"): (0.95, 2), (2, "BOR late"): (0.035, 3),
    (2, "PXP M2 local M0-M3 only (early)"): (0.90, 2), (2, "PXP M3 global M0-M3 only (late)"): (0.97, 2),
    (2, "early M2-M3 mean dLL"): (18.8, 1), (2, "early M2-M1 mean dLL"): (4.4, 1), (2, "early M2-M0 mean dLL"): (2.2, 1),
    (2, "late M3-M2 mean dLL"): (30.2, 1), (2, "late M3-M1 mean dLL"): (34.4, 1), (2, "late M3-M0 mean dLL"): (1.5, 1),
    (2, "recovery M2 early (%)"): (100, 0), (2, "recovery M2 late (%)"): (90, 0), (2, "recovery M3 early (%)"): (100, 0), (2, "recovery M3 late (%)"): (100, 0),
    (2, "cond-mean r local window local learner"): (0.99, 2), (2, "cond-mean r local window global learner"): (0.48, 2),
    (2, "cond-mean r global window global learner"): (0.99, 2), (2, "cond-mean r global window local learner"): (0.28, 2),
    (2, "binned slope t local raw"): (-5.4, 1), (2, "binned slope t global raw"): (5.2, 1),
    (2, "binned slope t local within"): (-1.8, 1), (2, "binned slope t global within"): (5.0, 1),
    (2, "single-trial slope t local own learner, within condition"): (-2.7, 1), (2, "single-trial slope t global own learner, within condition"): (6.1, 1),
    (2, "single-trial slope t local other learner beyond own"): (-2.6, 1), (2, "single-trial slope t global other learner beyond own"): (0.2, 1),
    (2, "clocks Wilcoxon p"): (0.055, 3), (2, "reset test1 BF01"): (3.8, 1), (2, "reset group-best rho"): (0.0, 1),
    (3, "eLORETA cluster p"): (0.0005, 4), (3, "eLORETA n sources"): (1153, 0), (3, "eLORETA peak |t|"): (10.2, 1),
    (3, "transversetemporal-lh fraction significant"): (0.73, 2), (3, "superiortemporal-lh fraction significant"): (0.80, 2),
    (3, "superiortemporal-rh fraction significant"): (0.13, 2),
    (3, "auditory cortex lh t"): (-7.5, 1), (3, "auditory cortex rh t"): (-2.7, 1),
    (3, "frontal latency jackknife t"): (-1.0, 1), (3, "leakage ratio (%)"): (33, 0), (3, "global n sources"): (278, 0),
}
rep = pd.DataFrame(REPORT).drop_duplicates(["section", "quantity"], keep="last")
rows = []
for (sec, q), (v, dec) in DRAFT.items():
    hit = rep[(rep.section == sec) & (rep.quantity == q)]
    nb = float(hit.notebook.iloc[0]) if len(hit) else np.nan
    rows.append(dict(section=sec, quantity=q, draft=v, notebook=round(nb, dec + 1) if np.isfinite(nb) else nb,
                     match=bool(np.isfinite(nb) and abs(round(nb, dec) - v) <= 10 ** -dec * 1.01)))
summary = pd.DataFrame(rows)
pd.set_option("display.width", 200, "display.max_rows", 200)
print(summary.to_string(index=False))
print(f"\n{summary.match.sum()}/{len(summary)} Results numbers reproduced at the reported precision")

## 7. Figure inventory

Every figure function run in this session, whether it succeeded, and the files now in `figures/`.

In [ ]:
print(pd.DataFrame(FIG_LOG, columns=["figure", "status", "seconds"]).to_string(index=False))
inv = sorted((PROJECT_ROOT / "figures").glob("*.pdf"))
print(f"\n{len(inv)} PDF figures in figures/ (each also as .png, most as .svg):")
for p in inv:
    print(f"  {p.name:34s} {time.strftime('%Y-%m-%d %H:%M', time.localtime(p.stat().st_mtime))}")